In [ ]:
import os
from pathlib import Path
from typing import Optional, Union

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import ticker as mtick

_SUPP_PLOT_RC = {
    "figure.dpi": 220,
    "axes.grid": False,
    "grid.alpha": 0.0,
    "grid.linewidth": 0.0,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "legend.title_fontsize": 11,
    "xtick.bottom": True,
    "ytick.left": True,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
}
mpl.rcParams.update(_SUPP_PLOT_RC)

def _merge_supp_rc(rc=None):
    merged = dict(_SUPP_PLOT_RC)
    if rc:
        merged.update(rc)
    return merged

if not hasattr(sns, "_lore_orig_set_theme"):
    sns._lore_orig_set_theme = sns.set_theme
if not hasattr(sns, "_lore_orig_set"):
    sns._lore_orig_set = sns.set

_ORIG_SNS_SET_THEME = sns._lore_orig_set_theme
_ORIG_SNS_SET = sns._lore_orig_set

def _supp_set_theme(*args, **kwargs):
    kwargs["rc"] = _merge_supp_rc(kwargs.get("rc"))
    return _ORIG_SNS_SET_THEME(*args, **kwargs)

def _supp_set(*args, **kwargs):
    kwargs["rc"] = _merge_supp_rc(kwargs.get("rc"))
    return _ORIG_SNS_SET(*args, **kwargs)

sns.set_theme = _supp_set_theme
sns.set = _supp_set
sns.set_theme(style="white", context="notebook", rc={"axes.grid": False, "grid.alpha": 0.0, "grid.linewidth": 0.0})

def _ensure_clear_figure(fig):
    for ax in fig.get_axes():
        try:
            ax.grid(False)
            ax.xaxis.grid(False, which="both")
            ax.yaxis.grid(False, which="both")
            ax.xaxis.set_ticks_position("bottom")
            ax.yaxis.set_ticks_position("left")
            ax.tick_params(axis="both", which="major", direction="out", length=4, width=1, bottom=True, left=True, top=False, right=False)
            ax.tick_params(axis="both", which="minor", direction="out", length=2.5, width=0.8, bottom=True, left=True, top=False, right=False)
        except Exception:
            pass

if not hasattr(plt, "_lore_orig_show"):
    plt._lore_orig_show = plt.show

_ORIG_PLT_SHOW = plt._lore_orig_show

def _show_clear(*args, **kwargs):
    for fig_num in plt.get_fignums():
        _ensure_clear_figure(plt.figure(fig_num))
    return _ORIG_PLT_SHOW(*args, **kwargs)

plt.show = _show_clear

def _print_donor_count(label, donor_count):
    print(f"{label}: n = {int(donor_count)}")

STAGE_ORDER = [
    "Embryoblast",
    "Germ layer-specific",
    "Tissue-specific",
    "Adult-specific",
]

MUT_TYPES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]

MUT_COLORS = {
    "C>A": "#b2182b",
    "C>G": "#f781bf",
    "C>T": "#a015a5",
    "T>A": "#682aaf",
    "T>C": "#355fbb",
    "T>G": "#053061",
}

STAGE_COLORS = {
    "Embryoblast": "#FFE5CC",
    "Germ layer-specific": "#FFB366",
    "Tissue-specific": "#FF7F00",
    "Adult-specific": "#CC5500",
}

WEIGHTING_COLORS = {
    "Variant count": "#CFCFCF",
    "Supporting reads": "#4C78A8",
}


def complement(base: str) -> str:
    return {"A": "T", "T": "A", "C": "G", "G": "C"}.get(str(base).upper(), "N")


def to_pyrimidine_class(ref: str, alt: str) -> Optional[str]:
    ref = str(ref).upper()
    alt = str(alt).upper()
    if ref in {"A", "G"}:
        ref = complement(ref)
        alt = complement(alt)
    if ref in {"C", "T"} and alt in {"A", "C", "G", "T"} and ref != alt:
        return f"{ref}>{alt}"
    return None


def _load_stage_mutations(mut_csv: Union[str, Path], donor_whitelist=None) -> pd.DataFrame:
    df = pd.read_csv(mut_csv, low_memory=False)
    if "stage_label" not in df.columns:
        raise ValueError(f"{mut_csv} is missing stage_label.")

    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].astype(str).isin(donor_whitelist)].copy()

    alt_col = "Base_observed" if "Base_observed" in df.columns else "ALT_expected"
    df = df[
        df["stage_label"].isin(STAGE_ORDER)
        & df["REF"].notna()
        & df[alt_col].notna()
    ].copy()

    df["mut_class"] = df.apply(lambda r: to_pyrimidine_class(r["REF"], r[alt_col]), axis=1)
    df = df[df["mut_class"].isin(MUT_TYPES)].copy()
    return df


def summarize_stage_spectrum_within_stage(mut_csv: Union[str, Path], donor_whitelist=None) -> pd.DataFrame:
    df = _load_stage_mutations(mut_csv, donor_whitelist=donor_whitelist)
    counts = (
        df.groupby(["donor", "stage_label", "mut_class"], observed=True)
        .size()
        .rename("n_mut")
        .reset_index()
    )
    counts["percent"] = (
        counts["n_mut"]
        / counts.groupby(["donor", "stage_label"], observed=True)["n_mut"].transform("sum")
        * 100.0
    )
    return counts


def summarize_stage_contribution_by_mut_class(mut_csv: Union[str, Path], donor_whitelist=None) -> pd.DataFrame:
    df = _load_stage_mutations(mut_csv, donor_whitelist=donor_whitelist)
    counts = (
        df.groupby(["donor", "mut_class", "stage_label"], observed=True)
        .size()
        .rename("n_mut")
        .reset_index()
    )
    counts["percent"] = (
        counts["n_mut"]
        / counts.groupby(["donor", "mut_class"], observed=True)["n_mut"].transform("sum")
        * 100.0
    )
    return counts


def summarize_stage_weighting(mut_csv: Union[str, Path], donor_whitelist=None) -> pd.DataFrame:
    df = _load_stage_mutations(mut_csv, donor_whitelist=donor_whitelist)

    counts = (
        df.groupby(["donor", "stage_label"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=STAGE_ORDER, fill_value=0.0)
    )
    count_pct = counts.div(counts.sum(axis=1), axis=0).replace([np.inf, -np.inf], np.nan) * 100.0
    count_pct = count_pct.dropna(how="all").reset_index().assign(weighting="Variant count")

    df["Num_reads"] = pd.to_numeric(df.get("Num_reads"), errors="coerce").fillna(0.0)
    read_counts = (
        df.groupby(["donor", "stage_label"], observed=True)["Num_reads"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=STAGE_ORDER, fill_value=0.0)
    )
    read_pct = (
        read_counts.div(read_counts.sum(axis=1), axis=0).replace([np.inf, -np.inf], np.nan) * 100.0
    )
    read_pct = read_pct.dropna(how="all").reset_index().assign(weighting="Supporting reads")

    long_df = pd.concat([count_pct, read_pct], ignore_index=True)
    return long_df.melt(
        id_vars=["donor", "weighting"],
        value_vars=STAGE_ORDER,
        var_name="Stage",
        value_name="Percent",
    )


def plot_stage_mutation_spectrum_lines(
    mut_csv: Union[str, Path],
    ax=None,
    title: Optional[str] = None,
    donor_whitelist=None,
) -> pd.DataFrame:
    counts = summarize_stage_spectrum_within_stage(mut_csv, donor_whitelist=donor_whitelist)
    pivoted = (
        counts.pivot_table(
            index=["donor", "stage_label"],
            columns="mut_class",
            values="percent",
            fill_value=0.0,
        )
        .reindex(columns=MUT_TYPES, fill_value=0.0)
        .reset_index()
    )

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6), dpi=1000)

    for mut_class in MUT_TYPES:
        for _, sub in pivoted.groupby("donor", observed=True):
            y = sub.set_index("stage_label").reindex(STAGE_ORDER)[mut_class].values
            ax.plot(STAGE_ORDER, y, color=MUT_COLORS[mut_class], alpha=0.18, linewidth=0.8)

        mean_vals = pivoted.groupby("stage_label", observed=True)[mut_class].mean().reindex(STAGE_ORDER).values
        ax.plot(STAGE_ORDER, mean_vals, color=MUT_COLORS[mut_class], label=mut_class, linewidth=2.5)

    ax.set_xlabel("Developmental Stage", fontsize=16)
    ax.set_ylabel("Percent of Mutations per Stage", fontsize=16)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(mtick.MultipleLocator(10))
    ax.yaxis.set_minor_locator(mtick.MultipleLocator(5))
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=14)
    ax.tick_params(axis="x", rotation=35)
    if title:
        ax.set_title(title, fontsize=16)
    return pivoted


def plot_stage_contribution_by_mut_class(
    mut_csv: Union[str, Path],
    ax=None,
    title: Optional[str] = None,
    donor_whitelist=None,
) -> pd.DataFrame:
    counts = summarize_stage_contribution_by_mut_class(mut_csv, donor_whitelist=donor_whitelist)
    stage_summary = (
        counts.pivot_table(
            index=["donor", "mut_class"],
            columns="stage_label",
            values="percent",
            fill_value=0.0,
        )
        .reindex(columns=STAGE_ORDER, fill_value=0.0)
        .reset_index()
    )

    mean_summary = (
        stage_summary.groupby("mut_class", observed=True)[STAGE_ORDER]
        .mean()
        .reindex(MUT_TYPES, fill_value=0.0)
    )

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6), dpi=1000)

    bottom = np.zeros(len(mean_summary), dtype=float)
    x = np.arange(len(mean_summary.index))
    for stage in STAGE_ORDER:
        vals = mean_summary[stage].to_numpy(float)
        ax.bar(
            x,
            vals,
            bottom=bottom,
            color=STAGE_COLORS[stage],
            edgecolor="black",
            linewidth=0.4,
            label=stage,
        )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(mean_summary.index, fontsize=12)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Mean Stage Contribution per Mutation Class (%)", fontsize=14)
    ax.set_xlabel("Mutation Class", fontsize=14)
    ax.yaxis.set_major_locator(mtick.MultipleLocator(10))
    ax.yaxis.set_minor_locator(mtick.MultipleLocator(5))
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:
        ax.set_title(title, fontsize=15)
    return counts


def plot_stage_weighting_comparison(
    mut_csv: Union[str, Path],
    ax=None,
    title: Optional[str] = None,
    donor_whitelist=None,
) -> pd.DataFrame:
    long_df = summarize_stage_weighting(mut_csv, donor_whitelist=donor_whitelist)
    summary = (
        long_df.groupby(["weighting", "Stage"], observed=True)["Percent"]
        .mean()
        .reset_index()
    )

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6), dpi=1000)

    sns.barplot(
        data=summary,
        x="Stage",
        y="Percent",
        hue="weighting",
        order=STAGE_ORDER,
        hue_order=list(WEIGHTING_COLORS),
        palette=WEIGHTING_COLORS,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
    )

    ax.set_xlabel("Developmental Stage", fontsize=14)
    ax.set_ylabel("Mean Percent Contribution Across Donors", fontsize=14)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(mtick.MultipleLocator(10))
    ax.yaxis.set_minor_locator(mtick.MultipleLocator(5))
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=25)
    if title:
        ax.set_title(title, fontsize=15)
    return long_df


from collections import Counter
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
import scipy.spatial.distance as ssd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgba
from matplotlib.container import BarContainer
from matplotlib.patches import Patch, PathPatch, Wedge
from matplotlib.path import Path as MplPath
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy import sparse, stats
from scipy.spatial.distance import squareform
from scipy.stats import linregress, pearsonr, zscore
from sklearn.decomposition import PCA


def _locate_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data").exists() and (candidate / "Figures.ipynb").exists() and (candidate / "Supp.ipynb").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the Lore2026_Clean directory or one of its subdirectories.")


REPO_ROOT = _locate_repo_root()
TABMUR_DIR = REPO_ROOT / "data" / "TabMur"
TABSAP_DIR = REPO_ROOT / "data" / "TabSap"
TABMUR_AGG = TABMUR_DIR / "aggregates"
TABSAP_AGG = TABSAP_DIR / "aggregates"


def _env_path(name):
    value = os.environ.get(name)
    return Path(value).expanduser() if value else None


def _resolve_first_existing_path(candidates):
    for candidate in candidates:
        if candidate is None:
            continue
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return None


def _sig_stars(p):
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    if p < 1e-1:
        return "#"
    return "ns"


def _summarize_iqr(df_percent):
    out = {}
    for stage in STAGE_ORDER:
        vals = df_percent[stage].dropna().to_numpy()
        out[stage] = {
            "median": np.median(vals),
            "q1": np.percentile(vals, 25),
            "q3": np.percentile(vals, 75),
        }
    return pd.DataFrame(out).T


def _load_stage_percent(base_dir):
    df = pd.read_csv(
        Path(base_dir) / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        index_col=0,
    ).reindex(columns=STAGE_ORDER, fill_value=0.0)
    return df.div(df.sum(axis=1), axis=0) * 100.0


def _load_stage_timing_donors(base_dir):
    return {str(donor) for donor in _load_stage_percent(base_dir).index}


def _load_burden_summary(base_dir, summary_name, age_col, donor_whitelist=None):
    df = pd.read_csv(Path(base_dir) / summary_name)
    df = df[(df["min_callable_sites"] == 100_000_000) & (df["n_celltypes"] >= 5)].copy()
    df["donor"] = df["donor"].astype(str)
    if donor_whitelist is not None:
        donor_whitelist = {str(donor) for donor in donor_whitelist}
        df = df[df["donor"].isin(donor_whitelist)].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["donor", age_col, "weighted_burden_per_kb"])
    return df.rename(columns={"weighted_burden_per_kb": "burden_per_kb"})


def _plot_burden_vs_age(df, age_col, point_color, xlabel):
    x = df[age_col].to_numpy(float)
    y = df["burden_per_kb"].to_numpy(float)

    if np.unique(x).size > 1:
        r, p = pearsonr(x, y)
        line_x = np.linspace(x.min(), x.max(), 200)
        line_y = np.polyval(np.polyfit(x, y, 1), line_x)
        title = "r = {:.2f}, p = {:.3g} {}".format(r, p, _sig_stars(p))
    else:
        line_x = line_y = None
        title = "r = nan, p = nan ns"

    fig, ax = plt.subplots(figsize=(12, 6), dpi=1000)
    ax.scatter(df[age_col], df["burden_per_kb"], color=point_color, edgecolor="black", s=60, linewidth=0.3)
    if line_x is not None:
        ax.plot(line_x, line_y, color="black", linewidth=2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("Mutation burden per kilobase", fontsize=16)
    plt.setp(ax.get_xticklabels(), fontsize=14)
    plt.setp(ax.get_yticklabels(), fontsize=14)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _average_stage_percent(base_dir):
    return _load_stage_percent(base_dir).mean().reindex(STAGE_ORDER).round(2)


def _show_stage_pies(stage_series_a, stage_series_b, labels, titles):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), dpi=700)
    for ax, avg, title in zip(axes, [stage_series_a, stage_series_b], titles):
        ax.pie(
            avg.values,
            colors=[STAGE_COLORS[s] for s in avg.index],
            startangle=90,
            wedgeprops={"edgecolor": "black", "linewidth": 0.5},
        )
        ax.set_title(title, fontsize=14)
        ax.axis("equal")
    fig.legend(labels, title="Stage", bbox_to_anchor=(1.05, 0.5), loc="center left", frameon=False, fontsize=11, title_fontsize=12)
    plt.tight_layout()
    plt.show()


def _load_age_specific_pie(base_dir, slope, intercept, tmax):
    base_stages = list(STAGE_ORDER)
    all_stages = base_stages + ["Age-specific"]
    colors = {
        "Embryoblast": "#FFE3C2",
        "Germ layer-specific": "#FFB066",
        "Tissue-specific": "#FF7A1A",
        "Adult-specific": "#CC4D00",
        "Age-specific": "#5A5A5A",
    }

    df = pd.read_csv(
        Path(base_dir) / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        index_col=0,
    ).reindex(columns=base_stages, fill_value=0.0)

    tri = 0.5 * slope * tmax ** 2
    rect = intercept * tmax
    age_fraction = max(0.0, min(1.0, tri / (tri + rect)))

    expanded = df.copy()
    adult_vals = expanded["Adult-specific"].copy()
    expanded["Adult-specific"] = (1 - age_fraction) * adult_vals
    expanded["Age-specific"] = age_fraction * adult_vals
    avg = (expanded.div(expanded.sum(axis=1), axis=0) * 100.0).mean().reindex(all_stages).round(2)
    return avg, all_stages, colors


def _prepare_adult_specific_clonality(mut_csv, alias_map=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    if "CB" not in df.columns and "cell_barcode" in df.columns:
        df = df.rename(columns={"cell_barcode": "CB"})
    df = df[df["stage_label"] == "Adult-specific"].copy()
    df["CB"] = df["CB"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "": np.nan})
    df = df[df["CB"].notna()].copy()
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    if "var_id" in df.columns:
        df["__var_key__"] = df["var_id"].astype(str)
    elif {"#CHROM", "Start", "REF", "ALT_expected"}.issubset(df.columns):
        df["__var_key__"] = df[["#CHROM", "Start", "REF", "ALT_expected"]].astype(str).agg("|".join, axis=1)
    else:
        df["__var_key__"] = df.index.astype(str)
    return df


def _load_stage_tissue_heatmap(mut_csv, cmap_name, tissue_replace=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    df = df[df["stage_label"].isin(STAGE_ORDER) & df["tissue"].notna()].copy()
    if tissue_replace:
        df["tissue"] = (
            df["tissue"].astype(str).str.replace("_", " ", regex=False).str.strip().replace(tissue_replace)
        )
    else:
        df["tissue"] = df["tissue"].astype(str).str.replace("_", " ", regex=False).str.strip()
    df["SitesPerCell"] = pd.to_numeric(df["SitesPerCell"], errors="coerce")
    df = df[df["SitesPerCell"].notna() & (df["SitesPerCell"] > 0)].copy()
    df["var_id"] = df["var_id"].astype(str)

    uniq = df.drop_duplicates(subset=["donor", "tissue", "stage_label", "var_id"])
    per_donor_mut = (
        uniq.groupby(["donor", "tissue", "stage_label"], observed=True)["var_id"]
        .nunique()
        .rename("mutations")
        .reset_index()
    )
    per_donor_exp = (
        df.groupby(["donor", "tissue", "stage_label"], observed=True)["SitesPerCell"]
        .sum()
        .rename("exposure")
        .reset_index()
    )
    per_donor = per_donor_mut.merge(per_donor_exp, on=["donor", "tissue", "stage_label"], how="outer").fillna(0)
    per_donor = per_donor[per_donor["exposure"] >= 1_000_000].copy()
    per_donor["rate_per_kb"] = (per_donor["mutations"] / per_donor["exposure"]) * 1000.0
    if per_donor.empty:
        raise ValueError("No donor x tissue x stage bins passed the exposure threshold.")

    agg = per_donor.groupby(["tissue", "stage_label"], observed=True)["rate_per_kb"].mean().reset_index()
    rate_pivot = (
        agg.pivot(index="stage_label", columns="tissue", values="rate_per_kb")
        .reindex(index=list(reversed(STAGE_ORDER)))
        .fillna(0.0)
    )
    rate_pivot = rate_pivot[rate_pivot.sum(axis=0).sort_values(ascending=False).index]

    data = rate_pivot.to_numpy()
    n_rows, n_cols = data.shape
    height = min(max(0.35 * n_rows, 4.0), 20.0)
    width = min(max(0.25 * n_cols, 6.0), 30.0)

    fig, ax = plt.subplots(figsize=(width, height), dpi=1000)
    im = ax.imshow(data, aspect="auto", cmap=cmap_name, interpolation="nearest")
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(rate_pivot.columns.tolist(), rotation=90, fontsize=10)
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(rate_pivot.index.tolist(), fontsize=10)
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(False)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_xlabel("Tissue type", fontsize=12)
    ax.set_ylabel("Developmental stage", fontsize=12)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Mutations per kilobase", fontsize=12)
    cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontsize(10)

    plt.tight_layout()
    plt.show()


def _plot_tissue_cdf(mut_csv, alias_map=None):
    df = _prepare_adult_specific_clonality(mut_csv, alias_map=alias_map)
    x_values = np.arange(1, 51)
    cdf_data = {}
    for tissue in sorted(df["tissue"].dropna().unique()):
        sub = df[df["tissue"] == tissue]
        sub_u = sub.drop_duplicates(subset=["donor", "__var_key__", "CB"])
        per_variant = sub_u.groupby(["donor", "__var_key__"])["CB"].nunique().reset_index(name="n_cells")
        if per_variant.empty:
            continue
        counts = per_variant["n_cells"].to_numpy()
        hist, _ = np.histogram(counts, bins=np.arange(1, 52))
        cdf_data[tissue] = np.cumsum(hist) / len(counts)

    palette = sns.color_palette("tab20", len(cdf_data))
    tissue_colors = {t: palette[i] for i, t in enumerate(cdf_data)}

    fig, ax = plt.subplots(figsize=(10, 6), dpi=2000)
    for tissue, cdf in cdf_data.items():
        ax.plot(x_values, cdf, label=tissue, linewidth=1.5, color=tissue_colors[tissue])
    ax.spines["top"].set_visible(False)
    ax.set_xscale("log")
    ax.set_xlabel("Number of cells carrying the mutation (log scale)", fontsize=16)
    ax.set_ylabel("Cumulative fraction of variants", fontsize=16)
    ax.set_xlim(1, 50)
    ax.set_ylim(0, 1.0)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: str(int(x))))
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    plt.tight_layout()
    plt.show()


def _plot_tissue_mutation_spectra(mut_csv, drop_tissues=None, alias_map=None, aggregation="weighted"):
    df = pd.read_csv(mut_csv, low_memory=False)
    df = df[df["tissue"].notna()].copy()
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    df["mut_class"] = df.apply(lambda r: to_pyrimidine_class(r["REF"], r["Base_observed"]), axis=1)
    df = df[df["mut_class"].isin(MUT_TYPES)].copy()
    df = df[df["stage_label"].isin(STAGE_ORDER)].copy()
    if drop_tissues:
        df = df[~df["tissue"].isin(drop_tissues)].copy()

    donor_stage_counts = (
        df[["donor", "stage_label"]]
        .drop_duplicates()
        .groupby("donor", observed=True)["stage_label"]
        .nunique()
    )
    eligible_donors = donor_stage_counts[donor_stage_counts == len(STAGE_ORDER)].index
    df = df[df["donor"].isin(eligible_donors)].copy()
    if df.empty:
        print("[WARN] No donors with variants across all 4 stages for {}".format(mut_csv))
        return

    for stage in STAGE_ORDER:
        subdf = df[df["stage_label"] == stage].copy()
        if subdf.empty:
            continue

        counts = (
            subdf.groupby(["donor", "tissue", "mut_class"], observed=True)
            .size()
            .rename("n_mut")
            .reset_index()
        )
        totals = counts.groupby(["donor", "tissue"], observed=True)["n_mut"].sum().rename("total_mut").reset_index()
        counts = counts.merge(totals, on=["donor", "tissue"], how="left")
        counts["percent"] = (counts["n_mut"] / counts["total_mut"]) * 100.0

        if aggregation == "mean":
            summary = (
                counts.groupby(["tissue", "mut_class"], observed=True)["percent"]
                .mean()
                .reset_index()
            )
        elif aggregation == "weighted":
            summary = (
                counts.assign(weighted_percent=counts["percent"] * counts["n_mut"])
                .groupby(["tissue", "mut_class"], observed=True)[["weighted_percent", "n_mut"]]
                .sum()
                .reset_index()
            )
            summary["percent"] = summary["weighted_percent"] / summary["n_mut"]
        else:
            raise ValueError("Unsupported aggregation: {}".format(aggregation))

        spectrum = (
            summary.pivot(index="tissue", columns="mut_class", values="percent")
            .reindex(columns=MUT_TYPES)
            .fillna(0.0)
            .sort_index()
        )

        fig, ax = plt.subplots(figsize=(10, 6), dpi=1200)
        bottom = np.zeros(len(spectrum), dtype=float)
        for mut_class in MUT_TYPES:
            vals = spectrum[mut_class].to_numpy(float)
            ax.bar(
                spectrum.index,
                vals,
                bottom=bottom,
                color=MUT_COLORS[mut_class],
                label=mut_class,
                edgecolor="black",
                linewidth=0.3,
            )
            bottom += vals

        ax.set_title("Stage: {}".format(stage), fontsize=14)
        ax.set_ylabel("% of Mutations", fontsize=14)
        ax.set_xlabel("Tissue", fontsize=14)
        ax.set_ylim(0, 100)
        plt.xticks(rotation=90, ha="center", fontsize=10)
        ax.tick_params(axis="y", labelsize=12)
        ax.legend(title="Mutation Type", fontsize=10, title_fontsize=12, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        plt.tight_layout()
        plt.show()


def _show_best_gene_example_regression_tabmur():
    data_path = TABMUR_DIR / "expression_coupling" / "example_regression_best_gene.csv.gz"
    if not data_path.exists():
        print("[WARN] Skipping TabMur example regression; missing input:")
        print("  - {}".format(data_path.relative_to(REPO_ROOT)))
        return

    df = pd.read_csv(data_path)
    if df.empty:
        print("[WARN] Skipping TabMur example regression; cached data is empty.")
        return

    best_gene = str(df["gene"].iloc[0])
    x = zscore(df["expression"].to_numpy())
    y = zscore(df["mutation_rate"].to_numpy())
    slope, intercept, r, p, stderr = linregress(x, y)

    plt.figure(figsize=(6, 5), dpi=600)
    sns.regplot(
        x=x,
        y=y,
        scatter_kws=dict(s=12, alpha=0.5, color="lightgray"),
        line_kws=dict(linewidth=2, color="black"),
    )
    plt.xlabel("{} expression (z-scored)".format(best_gene))
    plt.ylabel("Mutation rate (z-scored)")
    plt.title("TabMur example gene: r = {:.2f}, p = {:.2e}".format(r, p))
    plt.tight_layout()
    plt.show()


def _load_slopes(slopes_path):
    df = pd.read_csv(slopes_path)
    if not {"r", "p"}.issubset(df.columns):
        raise SystemExit("[ERROR] slope file missing required columns")
    df["logp"] = -np.log10(df["p"].replace(0, np.nan))
    df["r_jitter"] = df["r"] + np.random.normal(0, 1e-4, size=len(df))
    df["logp_jitter"] = df["logp"] + np.random.normal(0, 1e-2, size=len(df))
    df["sig_class"] = "ns"
    df.loc[(df["p"] <= 0.05) & (df["r"] >= 0.1), "sig_class"] = "sig_pos"
    df.loc[(df["p"] <= 0.05) & (df["r"] <= -0.1), "sig_class"] = "sig_neg"
    return df


def _plot_slope_hist(slopes_path, color, title):
    df = _load_slopes(slopes_path)
    plt.figure(figsize=(7, 6), dpi=600)
    sns.histplot(df["r"], bins=100, color=color, edgecolor=None)
    sns.despine()
    plt.xlabel("Pearson r")
    plt.ylabel("Number of genes")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def _plot_slope_volcano(slopes_path, title):
    df = _load_slopes(slopes_path)
    palette = {"sig_pos": "red", "sig_neg": "blue", "ns": "lightgray"}
    plt.figure(figsize=(7, 6), dpi=600)
    sns.scatterplot(
        data=df,
        x="r_jitter",
        y="logp_jitter",
        hue="sig_class",
        palette=palette,
        edgecolor=None,
        alpha=0.6,
        s=15,
        legend=False,
    )
    plt.xlabel("Pearson r")
    plt.ylabel("-log10(p)")
    plt.title(title)
    sns.despine()
    ymax = df["logp_jitter"].replace([np.inf, -np.inf], np.nan).dropna().max() * 1.05
    plt.ylim(0, min(ymax, 500))
    plt.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=1)
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.tight_layout()
    plt.show()
    print("Significant positive (upregulated):", int((df["sig_class"] == "sig_pos").sum()))
    print("Significant negative (downregulated):", int((df["sig_class"] == "sig_neg").sum()))
    print("Not significant:", int((df["sig_class"] == "ns").sum()))


def _load_gsea(path):
    return pd.read_csv(path, sep="\t")


def _latest_gsea_report(base_dir, direction, dataset_label):
    matches = sorted(Path(base_dir).glob(f"gsea_report_for_na_{direction}_*.tsv"))
    if not matches:
        raise FileNotFoundError(f"No {dataset_label} {direction} GSEA report found under {base_dir}")
    return matches[-1]


def _plot_go_bar(neg_path, pos_path, title):
    df_neg = _load_gsea(neg_path)
    df_pos = _load_gsea(pos_path)
    df_neg_top = df_neg.nsmallest(10, "FDR q-val").copy()
    df_pos_top = df_pos.nsmallest(10, "FDR q-val").copy()
    df_neg_top["direction"] = "Negative"
    df_pos_top["direction"] = "Positive"
    df_plot = pd.concat([df_pos_top, df_neg_top], ignore_index=True).sort_values("NES")
    df_plot["NES_raw"] = df_plot["NES"].astype(float)
    max_abs_nes = max(1.0, float(df_plot["NES_raw"].abs().max()))
    df_plot["NES_scaled"] = df_plot["NES_raw"] / max_abs_nes
    colors = df_plot["direction"].map({"Positive": "red", "Negative": "blue"})
    fig, ax = plt.subplots(figsize=(20, 8), dpi=600)
    bars = ax.barh(df_plot["NAME"], df_plot["NES_scaled"], color=colors)
    sns.despine(ax=ax)
    ax.set_xlabel("Relative NES within species")
    ax.set_ylabel("")
    ax.set_title(title)
    ax.set_xlim(-1.08, 1.08)
    ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.axvline(0, color="black", linewidth=0.8)
    for bar, raw_nes in zip(bars, df_plot["NES_raw"]):
        x = bar.get_width()
        y = bar.get_y() + (bar.get_height() / 2)
        if x >= 0:
            ax.text(x + 0.03, y, f"{raw_nes:.2f}", va="center", ha="left", fontsize=9)
        else:
            ax.text(x - 0.03, y, f"{raw_nes:.2f}", va="center", ha="right", fontsize=9)
    legend_handles = [
        Patch(facecolor="red", label="Positive"),
        Patch(facecolor="blue", label="Negative"),
    ]
    ax.legend(handles=legend_handles, title="Enrichment", frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.subplots_adjust(left=0.34, right=0.97, top=0.92, bottom=0.08)
    plt.show()


def _collapse_terms(df, nes_col="NES", term_col="NAME"):
    tmp = df.copy()
    tmp["absNES"] = tmp[nes_col].abs()
    tmp = tmp.sort_values("absNES", ascending=False)
    tmp = tmp.drop_duplicates(subset=term_col, keep="first")
    return tmp.set_index(term_col)[nes_col]


def _plot_cross_species_go_heatmap():
    mur_neg = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
    mur_pos = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
    sap_neg = _latest_gsea_report(TABSAP_DIR / "expression_coupling", "neg", "TabSap")
    sap_pos = _latest_gsea_report(TABSAP_DIR / "expression_coupling", "pos", "TabSap")

    mur_pos_df = _load_gsea(mur_pos)[["NAME", "NES"]]
    mur_neg_df = _load_gsea(mur_neg)[["NAME", "NES"]]
    sap_pos_df = _load_gsea(sap_pos)[["NAME", "NES"]]
    sap_neg_df = _load_gsea(sap_neg)[["NAME", "NES"]]

    def top10(df, direction):
        return df.sort_values("NES", ascending=(direction == "down")).head(10)

    mur_up = top10(mur_pos_df, "up").assign(group="Mouse_UP")
    mur_down = top10(mur_neg_df, "down").assign(group="Mouse_DOWN")
    sap_up = top10(sap_pos_df, "up").assign(group="Human_UP")
    sap_down = top10(sap_neg_df, "down").assign(group="Human_DOWN")
    all_top = pd.concat([mur_up, mur_down, sap_up, sap_down], ignore_index=True)

    mur_nes = _collapse_terms(pd.concat([mur_pos_df, mur_neg_df], ignore_index=True))
    sap_nes = _collapse_terms(pd.concat([sap_pos_df, sap_neg_df], ignore_index=True))

    terms = all_top["NAME"]
    heat = pd.DataFrame(
        {
            "NES_mouse": mur_nes.reindex(terms),
            "NES_human": sap_nes.reindex(terms),
            "group": all_top["group"].values,
        },
        index=terms,
    )
    heat[["NES_mouse", "NES_human"]] = heat[["NES_mouse", "NES_human"]].fillna(0)
    heat["group"] = pd.Categorical(
        heat["group"],
        categories=["Human_UP", "Human_DOWN", "Mouse_UP", "Mouse_DOWN"],
        ordered=True,
    )
    heat = heat.sort_values("group", kind="stable")
    heat = heat[~heat.index.duplicated(keep="first")]

    cmap = LinearSegmentedColormap.from_list("darkblue_white_darkcoral", ["#3B6BA5", "#FFFFFF", "#D56A4A"])
    plt.figure(figsize=(13, max(6, 0.38 * heat.shape[0])))
    sns.heatmap(
        heat[["NES_mouse", "NES_human"]],
        cmap=cmap,
        center=0,
        linewidths=0.5,
        linecolor="gray",
        cbar_kws={"label": "NES"},
    )
    plt.title("Top 10 up/down GO terms per species")
    plt.xlabel("Species")
    plt.ylabel("GO term")
    plt.tight_layout()
    plt.show()


def _plot_cross_species_go_scatter():
    mur_neg = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv"
    mur_pos = TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv"
    sap_neg = _latest_gsea_report(TABSAP_DIR / "expression_coupling", "neg", "TabSap")
    sap_pos = _latest_gsea_report(TABSAP_DIR / "expression_coupling", "pos", "TabSap")

    mur_all = _collapse_terms(pd.concat([_load_gsea(mur_pos)[["NAME", "NES"]], _load_gsea(mur_neg)[["NAME", "NES"]]], ignore_index=True))
    sap_all = _collapse_terms(pd.concat([_load_gsea(sap_pos)[["NAME", "NES"]], _load_gsea(sap_neg)[["NAME", "NES"]]], ignore_index=True))
    overlap = sorted(set(mur_all.index) & set(sap_all.index))
    df = pd.DataFrame({"NES_mouse": mur_all.loc[overlap], "NES_human": sap_all.loc[overlap]}, index=overlap)

    def classify(row):
        if row["NES_mouse"] > 0 and row["NES_human"] > 0:
            return "Concordant up"
        if row["NES_mouse"] < 0 and row["NES_human"] < 0:
            return "Concordant down"
        return "Opposite"

    df["category"] = df.apply(classify, axis=1)
    strength = (df["NES_mouse"].abs() + df["NES_human"].abs()) / 2
    df["size"] = 20 + 60 * (strength / strength.max())

    plt.figure(figsize=(4.5, 4.5), dpi=600)
    for category, color, alpha in [
        ("Concordant up", "#D56A4A", 0.65),
        ("Concordant down", "#3B6BA5", 0.65),
        ("Opposite", "#777777", 0.40),
    ]:
        sub = df[df["category"] == category]
        plt.scatter(
            sub["NES_mouse"],
            sub["NES_human"],
            s=sub["size"] * 0.20,
            color=color,
            alpha=alpha,
            edgecolor="none",
            label=category,
        )

    plt.xlim(-7, 7)
    plt.ylim(-7, 7)
    plt.axhline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
    plt.axvline(0, linestyle="--", linewidth=0.8, color="#DDDDDD")
    plt.xlabel("NES (Mouse)", fontsize=11)
    plt.ylabel("NES (Human)", fontsize=11)
    plt.title("GO pathway enrichment across species", fontsize=12)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(False)
    plt.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.04, 0.5))
    sns.despine(top=True, right=True)
    ax = plt.gca()
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    plt.tight_layout()
    plt.show()


def _show_96class_spectrum():
    fasta = pysam.FastaFile(str(REPO_ROOT / "data" / "ref_genome" / "gencode_v41_ercc.fa"))
    subs = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
    bases = ["A", "C", "G", "T"]
    ordered = ["{}[{}]{}".format(pre, sub, post) for sub in subs for pre in bases for post in bases]

    def get_context(chrom, pos):
        try:
            seq = fasta.fetch(chrom, pos - 2, pos + 1).upper()
            return seq if len(seq) == 3 else None
        except Exception:
            return None

    def reverse_complement(triplet):
        comp = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
        return "".join(comp.get(b, "N") for b in reversed(triplet))

    def to_pyrimidine(ref, alt, context):
        if ref in {"C", "T"}:
            return ref, alt, context
        comp = {"A": "T", "T": "A", "C": "G", "G": "C"}
        return comp[ref], comp[alt], reverse_complement(context)

    def plot_one(mut_csv):
        df = pd.read_csv(mut_csv)
        alt_col = "Base_observed" if "Base_observed" in df.columns else "ALT_expected"
        df = df[df["REF"].notna() & df[alt_col].notna()].copy()
        df["REF"] = df["REF"].str.upper()
        df["ALT"] = df[alt_col].str.upper()
        df = df[df["REF"] != df["ALT"]]
        df = df[df["REF"].isin(bases) & df["ALT"].isin(bases)]
        counts = Counter()
        for _, row in df.iterrows():
            chrom = row["#CHROM"] if "#CHROM" in row else row.get("CHROM")
            pos = int(row["Start"])
            context = get_context(chrom, pos)
            if context is None or len(context) != 3 or context[1] != row["REF"]:
                continue
            ref_norm, alt_norm, context_norm = to_pyrimidine(row["REF"], row["ALT"], context)
            sub = "{}>{}".format(ref_norm, alt_norm)
            counts["{}[{}]{}".format(context_norm[0], sub, context_norm[2])] += 1
        series = pd.Series({k: counts.get(k, 0) for k in ordered})
        total = series.sum()
        return series / total if total > 0 else series

    color_map = {
        "C>A": "#b2182b",
        "C>G": "#f781bf",
        "C>T": "#a015a5",
        "T>A": "#682aaf",
        "T>C": "#355fbb",
        "T>G": "#053061",
    }

    for dataset, mut_csv in [("TabMur", TABMUR_AGG / "mut_table.csv"), ("TabSap", TABSAP_AGG / "mut_table.csv")]:
        series = plot_one(mut_csv)
        plt.figure(figsize=(18, 4.8), dpi=1200)
        colors = [color_map[label.split("[")[1].split("]")[0]] for label in series.index]
        plt.bar(range(96), series.values, color=colors, width=0.8)
        plt.xticks(range(96), series.index, rotation=90, fontsize=7)
        plt.ylabel("Distribution", fontsize=12)
        plt.xlabel("Single Base Substitution Type", fontsize=12)
        ax = plt.gca()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[sub]) for sub in subs]
        ax.legend(handles, subs, title="Substitution", loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
        plt.tight_layout()
        plt.show()

    fasta.close()


def _show_expression_violin():
    def categorize_expression(series):
        return pd.qcut(np.log10(series + 1), q=3, labels=["Low", "Medium", "High"])

    obs = pd.read_csv(TABMUR_DIR / "single_cell_metadata" / "adata_obs.csv", low_memory=False)
    mut = pd.read_csv(TABMUR_AGG / "cb_mutation_rates__all.csv")
    obs["CB_stripped"] = obs["cell"].str[-16:]
    obs["merge_key"] = obs["mouse.id"].astype(str) + "_" + obs["CB_stripped"]
    mut["merge_key"] = mut["donor"].astype(str) + "_" + mut["CB"]
    tabmur = mut.merge(obs, on="merge_key")
    tabmur["expr_bin"] = categorize_expression(tabmur["n_counts"])
    tabmur["log10_mutation_rate"] = np.log10(tabmur["mutation_rate"] + 1e-6)

    barcode_map = pd.read_csv(TABSAP_DIR / "single_cell_metadata" / "cellbarcodes_clean.csv")
    obs = pd.read_csv(TABSAP_DIR / "single_cell_metadata" / "adata_obs.csv", low_memory=False)
    mut = pd.read_csv(TABSAP_AGG / "cb_mutation_rates__all.csv")
    barcode_map["CB_stripped"] = barcode_map["CB"].str[-16:]
    barcode_map["merge_key"] = barcode_map["donor"].astype(str) + "_" + barcode_map["CB_stripped"]
    mut["merge_key"] = mut["donor"].astype(str) + "_" + mut["CB"]
    tabsap = mut.merge(barcode_map, on="merge_key")
    tabsap = tabsap.merge(obs[["n_genes_by_counts"]], left_on="row_idx", right_index=True)
    tabsap["expr_bin"] = categorize_expression(tabsap["n_genes_by_counts"])
    tabsap["log10_mutation_rate"] = np.log10(tabsap["mutation_rate"] + 1e-6)

    expr_bins = ["Low", "Medium", "High"]
    blue_palette = [to_hex(plt.get_cmap("Blues")(0.3 + 0.6 * (i / max(1, len(expr_bins) - 1)))) for i in range(len(expr_bins))]
    green_palette = [to_hex(plt.get_cmap("Greens")(i / (len(expr_bins) - 1))) for i in range(len(expr_bins))]

    for data, title, palette in [
        (tabmur[["log10_mutation_rate", "expr_bin"]], "Mice per cell mutation burden across cell expression", blue_palette),
        (tabsap[["log10_mutation_rate", "expr_bin"]], "Humans per cell mutation burden across cell expression", green_palette),
    ]:
        plt.figure(figsize=(8.5, 6), dpi=800)
        sns.violinplot(x="expr_bin", y="log10_mutation_rate", data=data, inner="quartile", cut=0, hue="expr_bin", palette=palette, legend=False)
        plt.ylabel("Mutation Rate (Log Scale)")
        plt.xlabel("Expression")
        plt.title(title)
        sns.despine(top=True, right=True)
        plt.tight_layout()
        plt.show()


def _plot_tissue_pca(mut_csv, title, keep_tissues=None, alias_map=None):
    df = pd.read_csv(mut_csv, low_memory=False)
    df = df[df["tissue"].notna()].copy()
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    if keep_tissues:
        df = df[df["tissue"].isin(keep_tissues)].copy()
    df["mut_class"] = df.apply(lambda r: to_pyrimidine_class(r["REF"], r["Base_observed"]), axis=1)
    df = df[df["mut_class"].isin(MUT_TYPES)].copy()
    counts = (
        df.groupby(["donor", "tissue", "mut_class"], observed=True)
        .size()
        .rename("n_mut")
        .reset_index()
    )
    totals = counts.groupby(["donor", "tissue"], observed=True)["n_mut"].sum().rename("total").reset_index()
    counts = counts.merge(totals, on=["donor", "tissue"], how="left")
    counts["fraction"] = counts["n_mut"] / counts["total"]
    agg = counts.groupby(["tissue", "mut_class"], observed=True)["fraction"].mean().reset_index()
    spectrum = (
        agg.pivot(index="tissue", columns="mut_class", values="fraction")
        .reindex(columns=MUT_TYPES)
        .fillna(0.0)
    )
    labels = spectrum.index.tolist()
    cmap = plt.get_cmap("tab20", len(labels))
    colors = {t: cmap(i) for i, t in enumerate(labels)}
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(spectrum.values)
    fig, ax = plt.subplots(figsize=(8, 7), dpi=2000)
    for i, tissue in enumerate(labels):
        ax.scatter(coords[i, 0], coords[i, 1], s=80, color=colors[tissue], alpha=0.9, edgecolor="black", linewidth=0.3)
    ax.set_xlabel("PC1 ({:.1f}%)".format(pca.explained_variance_ratio_[0] * 100), fontsize=14)
    ax.set_ylabel("PC2 ({:.1f}%)".format(pca.explained_variance_ratio_[1] * 100), fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    handles = [
        Patch(facecolor=colors[tissue], edgecolor="black", linewidth=0.3, label=tissue)
        for tissue in labels
    ]
    ax.legend(
        handles=handles,
        title="Tissue",
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        frameon=False,
        fontsize=10,
        title_fontsize=11,
    )
    if keep_tissues:
        ax.set_aspect("equal", adjustable="datalim")
    plt.tight_layout(rect=(0, 0, 0.8, 1))
    plt.show()


def _plot_chord(go_path, title, alias_map=None, with_group_labels=True):
    df = pd.read_csv(go_path, sep="\t")
    if alias_map:
        df["tissue"] = df["tissue"].replace(alias_map)
    df["group"] = df["tissue"].astype(str) + "|" + df["cell_type"]
    df = df.sort_values(["group", "FDR"], ascending=[True, True])
    mat = df.pivot_table(index="go_term", columns="group", values="NES", aggfunc="mean")
    mat = mat.loc[(~mat.isna()).sum(axis=1) >= 5].fillna(0.0)
    corr = 1 - squareform(ssd.pdist(mat.T, metric="correlation"))
    corr_df = pd.DataFrame(corr, index=mat.columns, columns=mat.columns)
    unique_tissues = sorted(set(g.split("|")[0] for g in mat.columns))
    palette = sns.color_palette("tab20", len(unique_tissues))
    tissue2color = dict(zip(unique_tissues, palette))

    real_groups = sorted(mat.columns.tolist())
    edges = []
    for i, g1 in enumerate(real_groups):
        for j, g2 in enumerate(real_groups):
            if j <= i:
                continue
            t1, t2 = g1.split("|")[0], g2.split("|")[0]
            if t1 == t2:
                continue
            r = corr_df.loc[g1, g2]
            if r >= 0.20:
                edges.append((g1, g2, r))

    used = sorted({g1 for g1, g2, _ in edges} | {g2 for g1, g2, _ in edges})
    if not used:
        fig, ax = plt.subplots(figsize=(10, 4.5), dpi=600)
        ax.axis("off")
        ax.text(0.5, 0.55, "No cross-tissue links met r >= 0.2", ha="center", va="center", fontsize=14)
        ax.text(0.5, 0.42, title, ha="center", va="center", fontsize=12)
        plt.tight_layout()
        plt.show()
        return

    group_index = {g: i for i, g in enumerate(used)}
    matrix = np.zeros((len(used), len(used)))
    for g1, g2, r in edges:
        if g1 in group_index and g2 in group_index:
            i, j = group_index[g1], group_index[g2]
            matrix[i, j] = matrix[j, i] = r

    strengths = matrix.sum(axis=1)
    if np.allclose(strengths.sum(), 0):
        strengths = np.ones(len(used))
    gap = 0.03
    arc_total = (2 * np.pi) - gap * len(used)
    arc_lengths = arc_total * (strengths / strengths.sum())
    angle_bounds = []
    cursor = np.pi / 2
    for span in arc_lengths:
        start = cursor
        end = cursor - span
        angle_bounds.append((start, end))
        cursor = end - gap

    inner_radius = 0.84
    outer_radius = 1.0
    label_radius = 1.38
    max_weight = matrix.max() if matrix.size else 0.0
    fig, ax = plt.subplots(figsize=(20, 20), dpi=500)

    for group, (start, end) in zip(used, angle_bounds):
        tissue, _ = group.split("|", 1)
        wedge = Wedge(
            (0, 0),
            outer_radius,
            np.degrees(end),
            np.degrees(start),
            width=outer_radius - inner_radius,
            facecolor=tissue2color[tissue],
            edgecolor="white",
            linewidth=1.0,
        )
        ax.add_patch(wedge)

    for i, g1 in enumerate(used):
        for j in range(i + 1, len(used)):
            weight = matrix[i, j]
            if weight <= 0:
                continue
            mid_i = 0.5 * sum(angle_bounds[i])
            mid_j = 0.5 * sum(angle_bounds[j])
            p0 = np.array([inner_radius * np.cos(mid_i), inner_radius * np.sin(mid_i)])
            p3 = np.array([inner_radius * np.cos(mid_j), inner_radius * np.sin(mid_j)])
            p1 = p0 * 0.25
            p2 = p3 * 0.25
            codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4]
            path = MplPath([p0, p1, p2, p3], codes)
            c1 = np.array(to_rgba(tissue2color[g1.split("|")[0]]))
            c2 = np.array(to_rgba(tissue2color[g2.split("|")[0]]))
            edge_color = tuple(((c1 + c2) / 2)[:3]) + (0.45,)
            patch = PathPatch(
                path,
                facecolor="none",
                edgecolor=edge_color,
                linewidth=0.8 + (5.0 * weight / max_weight if max_weight > 0 else 0.0),
                capstyle="round",
            )
            ax.add_patch(patch)

    for group, (start, end) in zip(used, angle_bounds):
        angle = 0.5 * (start + end)
        tissue, celltype = group.split("|", 1)
        deg = np.degrees(angle)
        rotation = deg + 180 if 90 < deg < 270 else deg
        ha = "right" if 90 < deg < 270 else "left"
        label = "{}\n({})".format(celltype, tissue) if with_group_labels else celltype
        ax.text(
            label_radius * np.cos(angle),
            label_radius * np.sin(angle),
            label,
            rotation=rotation,
            rotation_mode="anchor",
            ha=ha,
            va="center",
            fontsize=8.5 if with_group_labels else 10,
            color=tissue2color[tissue] if with_group_labels else "black",
            weight="bold" if with_group_labels else None,
        )

    ax.set_title(title, fontsize=15, weight="bold" if with_group_labels else None)
    ax.set_xlim(-1.65, 1.65)
    ax.set_ylim(-1.65, 1.65)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    fig_legend, ax_legend = plt.subplots(figsize=(4.5, 6.5), dpi=400)
    ax_legend.axis("off")
    handles = [Patch(facecolor=tissue2color[t], edgecolor="black", linewidth=0.5, label=t) for t in sorted({g.split("|")[0] for g in used})]
    ax_legend.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.0, 0.5),
        ncol=1,
        fontsize=11,
        title="Tissue",
        title_fontsize=12,
        frameon=False,
        handlelength=1.5,
        handleheight=1.2,
    )
    plt.tight_layout()
    plt.show()


def _plot_clustermap(go_path, title):
    df = pd.read_csv(go_path, sep="\t")
    if not {"go_term", "cell_type", "NES"}.issubset(df.columns):
        raise SystemExit("[ERROR] Input must contain go_term, cell_type, NES")
    mat_ct = df.groupby(["go_term", "cell_type"], observed=True)["NES"].mean().unstack(fill_value=np.nan)
    mat_ct = mat_ct.loc[mat_ct.notna().sum(axis=1) >= 5]
    corr = 1 - squareform(ssd.pdist(mat_ct.T.fillna(0), metric="correlation"))
    corr_df = pd.DataFrame(corr, index=mat_ct.columns, columns=mat_ct.columns).replace([np.inf, -np.inf], np.nan)

    cmap = LinearSegmentedColormap.from_list("tabmur_redblue", ["blue", "white", "red"], N=256)
    cmap.set_bad(color="#d3d3d3")
    sns.set_context("notebook", font_scale=1.15)
    sns.set_style("white")
    plt.rcParams["figure.constrained_layout.use"] = False
    plt.rcParams["figure.autolayout"] = False
    grid = sns.clustermap(
        corr_df,
        cmap=cmap,
        center=0,
        figsize=(18, 16),
        xticklabels=True,
        yticklabels=True,
        cbar_pos=(1.05, 0.2, 0.03, 0.6),
        cbar_kws={},
    )
    grid.fig.set_dpi(350)
    plt.setp(grid.ax_heatmap.get_xticklabels(), rotation=90, ha="center", fontsize=9, color="black")
    plt.setp(grid.ax_heatmap.get_yticklabels(), rotation=0, fontsize=9, color="black")
    grid.fig.suptitle(title, x=0.02, y=1.02, ha="left", fontsize=14, weight="bold")
    cbar = grid.ax_cbar
    cbar.set_title("")
    cbar.set_xticklabels([])
    cbar.tick_params(size=0, labelsize=10)
    ordered_cell_types = grid.data2d.columns.tolist()
    plt.show()
    return ordered_cell_types





## Extended Data Fig. 1 - classification and workflow schematic

This panel is assembled as document artwork in `Files/20260611_Extended_Data.docx`; no executable plotting cell is present in this notebook.


## Extended Data Fig. 2 - developmental origin of somatic mutations


In [ ]:
df_percent = _load_stage_percent(TABMUR_AGG)
stage_df = df_percent.reset_index().rename(columns={"index": "donor"})
age = stage_df["donor"].astype(str).str.split("-").str[0].astype(float)
stage_df["AgeGroup"] = np.where(age <= 3, "Young (<=3 mo)", np.where(age >= 18, "Aged (>=18 mo)", None))
stage_df = stage_df.dropna(subset=["AgeGroup"])
melted = stage_df.melt(id_vars=["donor", "AgeGroup"], value_vars=STAGE_ORDER, var_name="Stage", value_name="Percent")
palette = {"Young (<=3 mo)": plt.get_cmap("Blues")(0.4), "Aged (>=18 mo)": plt.get_cmap("Blues")(0.9)}
fig, ax = plt.subplots(figsize=(8, 6), dpi=1000)
sns.boxplot(data=melted, x="Stage", y="Percent", hue="AgeGroup", order=STAGE_ORDER, palette=palette, fliersize=0, ax=ax, zorder=1)
sns.stripplot(data=melted, x="Stage", y="Percent", hue="AgeGroup", order=STAGE_ORDER, palette=palette, dodge=True, jitter=True, marker="o", size=5, alpha=0.8, edgecolor="black", linewidth=0.3, ax=ax, zorder=2)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Age Group", fontsize=10, title_fontsize=12, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Developmental stage", fontsize=12)
ax.set_ylabel("Percent contribution (%)", fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=11)
plt.tight_layout()
plt.show()
print("Mouse: n = {}".format(stage_df["donor"].nunique()))
print("Mouse median/IQR:\n", _summarize_iqr(stage_df[STAGE_ORDER]))


In [ ]:
df_percent = _load_stage_percent(TABSAP_AGG)
melted = df_percent.reset_index().melt(id_vars="index", value_vars=STAGE_ORDER, var_name="Stage", value_name="Percent").rename(columns={"index": "Donor"})
donors = df_percent.index.tolist()
cmap = plt.get_cmap("Greens", len(donors))
donor_to_color = {donor: to_hex(cmap(i)) for i, donor in enumerate(donors)}
fig, ax = plt.subplots(figsize=(10, 6), dpi=1200)
sns.boxplot(data=melted, x="Stage", y="Percent", order=STAGE_ORDER, color="#E9F7E5", fliersize=0, ax=ax, zorder=1)
for donor in donors:
    sub = melted[melted["Donor"] == donor]
    ax.scatter([STAGE_ORDER.index(stage) for stage in sub["Stage"]], sub["Percent"], color=donor_to_color[donor], s=60, alpha=0.95, edgecolor="black", linewidth=0.3, label=donor, zorder=3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Developmental stage", fontsize=16)
ax.set_ylabel("Percent contribution (%)", fontsize=16)
plt.setp(ax.get_xticklabels(), fontsize=14, rotation=30, ha="right")
plt.setp(ax.get_yticklabels(), fontsize=14)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Donor", bbox_to_anchor=(1.02, 0.5), loc="center left", fontsize=11, title_fontsize=13, frameon=False)
ax.grid(False)
plt.tight_layout()
plt.subplots_adjust(right=0.8)
plt.show()
print("Human: n = {}".format(len(donors)))
print("Human median/IQR:\n", _summarize_iqr(df_percent))


## Figure 3 in Extended_Data.docx - orthogonal and external validation


In [ ]:
VALIDATION_DIR = REPO_ROOT / "data" / "validation"
PLOT_ORDER = ["Zygote", "Embryoblast", "Germ layer-specific", "Tissue-specific", "Adult-specific"]
VALIDATION_STAGE_ORDER = ["Embryoblast", "Germ layer-specific", "Tissue-specific", "Adult-specific"]
VALIDATION_DONOR_SUBSETS = {}
VALIDATION_LABELS = {
    "orth_val_1": "Orthogonal validation 1",
    "orth_val_2": "Orthogonal validation 2",
    "kim_val": "Kim validation",
    "moore_val": "Moore validation",
}

def _load_validation_curves(dataset):
    curves = pd.read_csv(VALIDATION_DIR / dataset / "developmental_timing_curves_by_donor.csv").set_index("donor").reindex(columns=PLOT_ORDER, fill_value=0.0)
    donor_subset = VALIDATION_DONOR_SUBSETS.get(dataset)
    if donor_subset is not None:
        curves = curves.loc[curves.index.intersection(donor_subset)]
    return curves

def _load_validation_stage_fraction(dataset):
    counts = (
        pd.read_csv(VALIDATION_DIR / dataset / "stage_counts_by_donor.csv")
        .set_index("donor")
        .reindex(columns=VALIDATION_STAGE_ORDER, fill_value=0.0)
    )
    donor_subset = VALIDATION_DONOR_SUBSETS.get(dataset)
    if donor_subset is not None:
        counts = counts.loc[counts.index.intersection(donor_subset)]
    frac = counts.div(counts.sum(axis=1), axis=0).replace([np.inf, -np.inf], np.nan)
    return frac.dropna(how="all")

def _main_stage_profile():
    mur = (_load_stage_percent(TABMUR_AGG).mean() / 100.0).reindex(VALIDATION_STAGE_ORDER)
    sap = (_load_stage_percent(TABSAP_AGG).mean() / 100.0).reindex(VALIDATION_STAGE_ORDER)
    return pd.concat([mur.rename("TabMur"), sap.rename("TabSap")], axis=1).mean(axis=1)

datasets_to_plot = ["orth_val_1"]
titles = ["Orthogonal validation 1"]

validation_donor_counts = {}

fig, axes = plt.subplots(1, 1, figsize=(8, 6), dpi=600, sharey=True)
axes = [axes]
for ax, dataset, title in zip(axes, datasets_to_plot, titles):
    curves = _load_validation_curves(dataset)
    n_donors = len(curves)
    validation_donor_counts[dataset] = n_donors
    if n_donors <= 10:
        cmap = plt.get_cmap("tab10", max(n_donors, 1))
        for idx, (donor, row) in enumerate(curves.iterrows()):
            ax.plot(PLOT_ORDER, row[PLOT_ORDER].to_numpy(float), marker="o", linewidth=2, color=cmap(idx), markeredgecolor="black", markeredgewidth=0.3, label=donor)
    else:
        for _, row in curves.iterrows():
            ax.plot(PLOT_ORDER, row[PLOT_ORDER].to_numpy(float), marker="o", linewidth=1.0, color="#9AA3B2", alpha=0.25)
        ax.plot(PLOT_ORDER, curves[PLOT_ORDER].mean(axis=0).to_numpy(float), marker="o", linewidth=3, color="#123A73", markeredgecolor="black", markeredgewidth=0.3, label="Mean")
    ax.set_title(title)
    ax.set_xlabel("Developmental stage")
    ax.set_ylabel("Cumulative fraction of staged variants")
    ax.set_ylim(0, 1.05)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
axes[0].legend(frameon=False, bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.tight_layout()
plt.subplots_adjust(right=0.84)
plt.show()
for dataset in datasets_to_plot:
    _print_donor_count(VALIDATION_LABELS.get(dataset, dataset), validation_donor_counts[dataset])

print("Main manuscript mean stage profile (%)")
print((_main_stage_profile() * 100).round(2).to_string())
for dataset in datasets_to_plot:
    print(f"\n## {dataset} developmental timing curves")
    print(_load_validation_curves(dataset).round(3).to_string())


In [ ]:
plt.style.use(str(REPO_ROOT / "styles" / "my_style.mplstyle"))
df = pd.read_csv(REPO_ROOT / "data" / "validation" / "colon" / "filtered_mutations" / "all_cells_merged.csv")
meta = pd.read_csv(REPO_ROOT / "data" / "validation" / "colon" / "samples.txt", sep="\t", comment="#", names=["donor", "sample_id", "age", "origin", "sex"])
df = df.merge(meta[["sample_id", "donor", "origin"]], on="sample_id", how="left")
per_cell = df.groupby(["donor", "origin", "CB", "SitesPerCell"], as_index=False).agg(n_mutations=("is_mutated", "sum"))
per_cell["group"] = per_cell["origin"].map({"Normal": "Healthy", "Tumor": "Tumor", "Tumor-2": "Tumor"})
per_cell = per_cell.dropna(subset=["group"])
per_cell["mutation_rate"] = per_cell["n_mutations"] / per_cell["SitesPerCell"] * 1000.0
summary = per_cell.groupby(["donor", "group"], as_index=False)["mutation_rate"].agg(mean_rate="mean", sem_rate=lambda x: stats.sem(x, nan_policy="omit"))
mean_wide = summary.pivot(index="donor", columns="group", values="mean_rate").fillna(0).reindex(columns=["Healthy", "Tumor"], fill_value=0)
sem_wide = summary.pivot(index="donor", columns="group", values="sem_rate").fillna(0).reindex(columns=["Healthy", "Tumor"], fill_value=0)
fig, ax = plt.subplots(figsize=(12, 8), dpi=900)
mean_wide.plot.bar(ax=ax, yerr=sem_wide.T.to_numpy(), capsize=5, color=["#1f77b4", "#ff7f0e"], rot=0)
containers = [c for c in ax.containers if isinstance(c, BarContainer)]
bars_h = containers[0] if len(containers) >= 1 else None
bars_t = containers[1] if len(containers) >= 2 else None
donors = mean_wide.index
y_max = (mean_wide.add(sem_wide, fill_value=0)).to_numpy().max()
ax.set_ylim(-y_max * 0.1, y_max * 1.25)

def sig_marker(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

for i, donor in enumerate(donors):
    rates_h = per_cell.loc[(per_cell["donor"] == donor) & (per_cell["group"] == "Healthy"), "mutation_rate"]
    rates_t = per_cell.loc[(per_cell["donor"] == donor) & (per_cell["group"] == "Tumor"), "mutation_rate"]
    if len(rates_h) and len(rates_t):
        star = sig_marker(stats.ttest_ind(rates_h, rates_t, nan_policy="omit").pvalue)
        if star and bars_h is not None and bars_t is not None:
            bh = bars_h.patches[i]
            bt = bars_t.patches[i]
            x1 = bh.get_x() + bh.get_width() / 2
            x2 = bt.get_x() + bt.get_width() / 2
            y0 = max(bh.get_height(), bt.get_height())
            h = y_max * 0.04
            ax.plot([x1, x1, x2, x2], [y0 + h, y0 + 2 * h, y0 + 2 * h, y0 + h], lw=1.2, color="black")
            ax.text((x1 + x2) / 2, y0 + 2 * h + y_max * 0.01, star, ha="center", va="bottom", fontsize=14, color="black")
    if donor in {"SC035", "SC044"}:
        label = "MSI-H"
    elif donor in {"SC040", "SC041", "SC043"}:
        label = "MSS"
    else:
        label = None
    if label:
        ax.text(i, -y_max * 0.05, label, ha="center", va="top", fontsize=11, color="black", fontweight="bold")
ax.set_xticks(np.arange(len(donors)))
ax.set_xticklabels([""] * len(donors))
ax.set_ylabel("Average Mutation Rate (mut/kb per cell)", fontsize=14, color="black")
ax.set_xlabel("Donor", fontsize=14, color="black")
ax.get_legend().remove()
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


## Extended Data Fig. 4 - mutation spectra validation


In [ ]:
mur_timing_donors = _load_stage_timing_donors(REPO_ROOT / "data" / "TabMur" / "aggregates")
fig, ax = plt.subplots(figsize=(10, 6), dpi=1000)
plot_stage_mutation_spectrum_lines(
    REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
    ax=ax,
    title="Mouse (TabMur)",
    donor_whitelist=mur_timing_donors,
)
ax.legend(title="Mutation Type", fontsize=14, title_fontsize=14, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.subplots_adjust(right=0.8)
plt.show()
_print_donor_count("Mouse (TabMur)", len(mur_timing_donors))


In [ ]:
sap_timing_donors = _load_stage_timing_donors(TABSAP_AGG)
fig, ax = plt.subplots(figsize=(10, 6), dpi=1000)
plot_stage_mutation_spectrum_lines(
    TABSAP_AGG / "mut_table_with_stage.csv",
    ax=ax,
    title="Human (TabSap)",
    donor_whitelist=sap_timing_donors,
)
ax.legend(title="Mutation Type", fontsize=14, title_fontsize=14, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.subplots_adjust(right=0.8)
plt.show()
_print_donor_count("Human (TabSap)", len(sap_timing_donors))


In [ ]:
_show_96class_spectrum()


## Extended Data Fig. 5 - transcriptional activity and tissue PCA


In [ ]:
_show_expression_violin()


In [ ]:
_plot_tissue_pca(TABMUR_AGG / "mut_table_with_stage.csv", "PCA of tissue mutational spectra (TabMur)")


In [ ]:
_plot_tissue_pca(
    TABSAP_AGG / "mut_table_with_stage.csv",
    "PCA of tissue mutational spectra (TabSap, 10 tissues)",
    keep_tissues=["Bladder", "Blood", "Fat", "Limb_Muscle", "Liver", "Lung", "Lymph_Node", "Marrow", "Spleen", "Tongue"],
    alias_map={"Bone_Marrow": "Marrow", "BoneMarrow": "Marrow", "Muscle": "Limb_Muscle", "SkeletalMuscle": "Limb_Muscle", "LymphNode": "Lymph Node", "LN": "Lymph Node"},
)


In [ ]:
_plot_slope_hist(TABMUR_DIR / "expression_coupling" / "slopes_protein_coding.csv", "skyblue", "Mice")


In [ ]:
_plot_slope_hist(TABSAP_DIR / "expression_coupling" / "slopes_protein_coding.csv", "green", "Humans")


## Extended Data Fig. 6 - active-gene-span normalization


In [ ]:
                                                                          
                                                                              
                                                  
import re

ACTIVE_GENE_FRACTION_THRESHOLD = 0.10
ACTIVE_GENE_MIN_CELLS = 20
HOUSEKEEPING_TARGET_BP = 1_000_000
ACTIVE_STAGE_ORDER = [stage for stage in STAGE_ORDER if stage != "Zygote"]
ACTIVE_NORM_CONFIG = [
    {
        "species": "TabSap (human)",
        "mut_csv": TABSAP_AGG / "mut_table_with_stage.csv",
        "stage_csv": TABSAP_AGG / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        "group_slopes_csv": TABSAP_DIR / "expression_coupling_by_group" / "slopes_by_group_protein_coding.csv",
        "group_cells_csv": TABSAP_DIR / "expression_coupling" / "mutation_rate__celltype_specific.csv",
        "gtf_candidates": [REPO_ROOT / "data" / "ref_genome" / "gencode_v44.gtf", _env_path("LORE2026_HUMAN_GTF")],
        "housekeeping_slopes_csv": TABSAP_DIR / "expression_coupling" / "slopes_protein_coding.csv",
        "housekeeping_gene_col": "gene_symbol",
        "color": "#2E8B57",
    },
    {
        "species": "TabMur (mouse)",
        "mut_csv": REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
        "stage_csv": REPO_ROOT / "data" / "TabMur" / "aggregates" / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
        "group_slopes_csv": REPO_ROOT / "data" / "TabMur" / "expression_coupling_by_group" / "slopes_by_group_protein_coding.csv",
        "group_cells_csv": REPO_ROOT / "data" / "TabMur" / "expression_coupling" / "mutation_rate__celltype_specific.csv",
        "gtf_candidates": [TABMUR_DIR / "gencode.vM36.annotation.gtf", _env_path("LORE2026_MOUSE_GTF")],
        "housekeeping_slopes_csv": REPO_ROOT / "data" / "TabMur" / "expression_coupling" / "slopes_protein_coding.csv",
        "housekeeping_gene_col": "gene",
        "color": "#4C78A8",
    },
]
ACTIVE_STAGE_COLORS = {
    "Embryoblast": "#FFE5CC",
    "Germ layer-specific": "#FFB366",
    "Tissue-specific": "#FF7F00",
    "Adult-specific": "#CC5500",
}
ACTIVE_GENE_NAME_RE = re.compile(r'gene_name \"([^\"]+)\"')
ACTIVE_GENE_TYPE_RE = re.compile(r'gene_type \"([^\"]+)\"|gene_biotype \"([^\"]+)\"')

def _canon_group_label(value):
    text = str(value).strip().lower().replace("&", "and")
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def _load_protein_coding_gene_lengths(gtf_path):
    rows = []
    seen = set()
    with open(gtf_path) as handle:
        for line in handle:
            if line.startswith("#"):
                continue
            chrom, _, feature, start, end, _, _, _, attrs = line.rstrip("\n").split("\t")
            if feature != "gene":
                continue
            gene_match = ACTIVE_GENE_NAME_RE.search(attrs)
            if gene_match is None:
                continue
            gene_name = gene_match.group(1)
            if gene_name in seen:
                continue
            type_match = ACTIVE_GENE_TYPE_RE.search(attrs)
            gene_type = (type_match.group(1) or type_match.group(2)) if type_match else None
            if gene_type not in {"protein_coding", "protein-coding"}:
                continue
            start_i = int(start)
            end_i = int(end)
            rows.append({
                "gene": gene_name,
                "chrom": chrom,
                "length_bp": end_i - start_i + 1,
                "gene_key": str(gene_name),
            })
            seen.add(gene_name)
    return pd.DataFrame(rows)

def _housekeeping_reference_bp(slopes_csv, gene_lengths, gene_col):
    slopes = pd.read_csv(slopes_csv)
    key_col = gene_col if gene_col in slopes.columns else "gene"
    ranked = (
        slopes[[key_col, "n_cells_expr"]]
        .rename(columns={key_col: "gene"})
        .assign(gene=lambda df: df["gene"].astype(str))
        .merge(gene_lengths[["gene", "chrom", "length_bp"]], on="gene", how="inner")
    )
    ranked = ranked[~ranked["chrom"].isin(["chrM", "MT", "M", "chrMT"])].copy()
    ranked = ranked.sort_values(["n_cells_expr", "length_bp"], ascending=[False, True])
    ranked = ranked.drop_duplicates(subset="gene").reset_index(drop=True)
    stop_idx = ranked["length_bp"].cumsum().searchsorted(HOUSEKEEPING_TARGET_BP) + 1
    return int(ranked.iloc[:stop_idx]["length_bp"].sum())

def _compute_group_active_spans(group_slopes_csv, group_cells_csv, gene_lengths):
    group_slopes = pd.read_csv(group_slopes_csv)
    group_cells = pd.read_csv(group_cells_csv)
    group_slopes["tissue_key"] = group_slopes["tissue"].map(_canon_group_label)
    group_slopes["cell_type_key"] = group_slopes["cell_type"].map(_canon_group_label)
    group_cells["tissue_key"] = group_cells["tissue"].map(_canon_group_label)
    group_cells["cell_type_key"] = group_cells["cell_type"].map(_canon_group_label)
    group_sizes = (
        group_cells.groupby(["tissue_key", "cell_type_key"], observed=True)
        .size()
        .rename("n_group_cells")
        .reset_index()
    )
    group_slopes = group_slopes.merge(group_sizes, on=["tissue_key", "cell_type_key"], how="inner")
    group_slopes["expr_fraction"] = group_slopes["n_cells_expr"] / group_slopes["n_group_cells"]
    gene_col = "symbol" if "symbol" in group_slopes.columns else "gene"
    group_slopes["gene_key"] = group_slopes[gene_col].astype(str)
    group_slopes = group_slopes.merge(gene_lengths[["gene_key", "length_bp"]], on="gene_key", how="inner")
    active = group_slopes[
        (group_slopes["n_cells_expr"] >= ACTIVE_GENE_MIN_CELLS)
        & (group_slopes["expr_fraction"] >= ACTIVE_GENE_FRACTION_THRESHOLD)
    ].copy()
    cell_type_span = (
        active.groupby(["tissue_key", "cell_type_key"], observed=True)["length_bp"]
        .sum()
        .rename("cell_type_span_bp")
        .reset_index()
    )
    tissue_span = (
        active[["tissue_key", "gene_key", "length_bp"]]
        .drop_duplicates()
        .groupby("tissue_key", observed=True)["length_bp"]
        .sum()
        .rename("tissue_span_bp")
        .reset_index()
    )
    return cell_type_span, tissue_span

def _estimate_donor_stage_active_span(mut_csv, cell_type_span, tissue_span):
    mut = pd.read_csv(mut_csv, low_memory=False)
    mut = mut[mut["stage_label"].isin(ACTIVE_STAGE_ORDER)].copy()
    mut["tissue_key"] = mut["tissue"].map(_canon_group_label)
    mut["cell_type_key"] = mut["cell_type_stage"].map(_canon_group_label)
    cell_type_map = {(row.tissue_key, row.cell_type_key): row.cell_type_span_bp for row in cell_type_span.itertuples()}
    tissue_map = {row.tissue_key: row.tissue_span_bp for row in tissue_span.itertuples()}
    rows = []
    for (donor, stage), sub in mut.groupby(["donor", "stage_label"], observed=True):
        if stage in {"Tissue-specific", "Adult-specific"}:
            weights = sub.groupby(["tissue_key", "cell_type_key"], observed=True).size().rename("w").reset_index()
            weights["active_span_bp"] = weights.apply(
                lambda row: cell_type_map.get((row["tissue_key"], row["cell_type_key"]), np.nan), axis=1
            )
        elif stage == "Germ layer-specific":
            weights = sub.groupby("tissue_key", observed=True).size().rename("w").reset_index()
            weights["active_span_bp"] = weights["tissue_key"].map(tissue_map)
        else:
            weights = sub.groupby("germ_layer", observed=True).size().rename("w").reset_index()
            gl_spans = (
                sub[["germ_layer", "tissue_key"]]
                .drop_duplicates()
                .merge(tissue_span, on="tissue_key", how="left")
                .groupby("germ_layer", observed=True)["tissue_span_bp"]
                .mean()
                .rename("active_span_bp")
                .reset_index()
            )
            weights = weights.merge(gl_spans, on="germ_layer", how="left")
        valid = weights.dropna(subset=["active_span_bp"]).copy()
        span_bp = np.average(valid["active_span_bp"], weights=valid["w"]) if not valid.empty else np.nan
        rows.append({"donor": donor, "stage_label": stage, "active_span_bp": span_bp})
    return pd.DataFrame(rows)

active_norm_rows = []
active_norm_donor_tables = {}
for cfg in ACTIVE_NORM_CONFIG:
    gtf_path = _resolve_first_existing_path(cfg["gtf_candidates"])
    if gtf_path is None:
        candidates = ", ".join(str(path) for path in cfg["gtf_candidates"] if path is not None)
        raise FileNotFoundError(f"No GTF found for {cfg['species']}. Tried: {candidates}")
    print(f"{cfg['species']}: using GTF {gtf_path}")
    gene_lengths = _load_protein_coding_gene_lengths(gtf_path)
    housekeeping_bp = _housekeeping_reference_bp(
        cfg["housekeeping_slopes_csv"],
        gene_lengths,
        cfg["housekeeping_gene_col"],
    )
    cell_type_span, tissue_span = _compute_group_active_spans(
        cfg["group_slopes_csv"],
        cfg["group_cells_csv"],
        gene_lengths,
    )
    donor_stage_span = _estimate_donor_stage_active_span(cfg["mut_csv"], cell_type_span, tissue_span)
    stage_span_defaults = donor_stage_span.groupby("stage_label", observed=True)["active_span_bp"].mean()

    stage_burden = pd.read_csv(cfg["stage_csv"]).rename(columns={"Unnamed: 0": "donor"})
    common_stages = [stage for stage in ACTIVE_STAGE_ORDER if stage in stage_burden.columns]
    long = stage_burden[["donor", *common_stages]].melt(
        id_vars="donor",
        var_name="stage_label",
        value_name="burden_per_kb",
    )
    merged = long.merge(donor_stage_span, on=["donor", "stage_label"], how="left")
    merged["active_span_bp"] = merged.apply(
        lambda row: stage_span_defaults.get(row["stage_label"]) if pd.isna(row["active_span_bp"]) else row["active_span_bp"],
        axis=1,
    )
    merged["housekeeping_equivalent_score"] = merged["burden_per_kb"] * housekeeping_bp / merged["active_span_bp"]

    original_pct = (
        stage_burden.set_index("donor")[common_stages]
        .div(stage_burden.set_index("donor")[common_stages].sum(axis=1), axis=0)
        * 100.0
    )
    active_pct = (
        merged.pivot(index="donor", columns="stage_label", values="housekeeping_equivalent_score")[common_stages]
        .div(
            merged.pivot(index="donor", columns="stage_label", values="housekeeping_equivalent_score")[common_stages].sum(axis=1),
            axis=0,
        )
        * 100.0
    )
    active_norm_donor_tables[cfg["species"]] = active_pct.copy()

    for stage in common_stages:
        active_norm_rows.append({
            "species": cfg["species"],
            "stage_label": stage,
            "normalization": "Original standardized burden",
            "mean_percent": float(original_pct[stage].mean()),
            "mean_active_span_mb": float(stage_span_defaults.get(stage, np.nan) / 1_000_000),
            "housekeeping_panel_mb": housekeeping_bp / 1_000_000,
        })
        active_norm_rows.append({
            "species": cfg["species"],
            "stage_label": stage,
            "normalization": "Active-gene-span normalized",
            "mean_percent": float(active_pct[stage].mean()),
            "mean_active_span_mb": float(stage_span_defaults.get(stage, np.nan) / 1_000_000),
            "housekeeping_panel_mb": housekeeping_bp / 1_000_000,
        })

active_stage_normalization_summary = pd.DataFrame(active_norm_rows)
display(
    active_stage_normalization_summary.round({
        "mean_percent": 2,
        "mean_active_span_mb": 2,
        "housekeeping_panel_mb": 3,
    })
)

for cfg in ACTIVE_NORM_CONFIG:
    species = cfg["species"]
    fig, ax = plt.subplots(figsize=(6.5, 5), dpi=350)
    sub = active_stage_normalization_summary[active_stage_normalization_summary["species"] == species].copy()
    sns.barplot(
        data=sub,
        x="stage_label",
        y="mean_percent",
        hue="normalization",
        order=ACTIVE_STAGE_ORDER,
        hue_order=["Original standardized burden", "Active-gene-span normalized"],
        palette=["#D9D9D9", cfg["color"]],
        edgecolor="black",
        linewidth=0.5,
        ax=ax,
    )
    ax.set_title(species, fontsize=14)
    ax.set_xlabel("")
    ax.set_ylabel("Mean donor-level stage contribution (%)")
    ax.tick_params(axis="x", rotation=30)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    plt.tight_layout()
    plt.show()
for cfg in ACTIVE_NORM_CONFIG:
    _print_donor_count(cfg["species"], len(active_norm_donor_tables[cfg["species"]]))




## Extended Data Fig. 7 - substitution-class sensitivity


In [ ]:
base_change = pd.read_csv(REPO_ROOT / "data" / "revision" / "base_change_stage_sensitivity.csv")
base_change_donor_counts = base_change.groupby("species", observed=True)["donor"].nunique().to_dict()
summary = (
    base_change.groupby(["species", "mut_class", "stage_label"], observed=True)["variant_percent"]
    .mean()
    .reset_index()
)
for species in ["Mouse", "Human"]:
    fig, ax = plt.subplots(figsize=(8, 6), dpi=1000)
    sub = summary[summary["species"] == species]
    pivot = (
        sub.pivot(index="mut_class", columns="stage_label", values="variant_percent")
        .reindex(index=MUT_TYPES, columns=STAGE_ORDER, fill_value=0.0)
    )
    bottom = np.zeros(len(pivot), dtype=float)
    x = np.arange(len(pivot.index))
    for stage in STAGE_ORDER:
        vals = pivot[stage].to_numpy(float)
        ax.bar(
            x,
            vals,
            bottom=bottom,
            color=STAGE_COLORS[stage],
            edgecolor="black",
            linewidth=0.4,
            label=stage,
        )
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha="right")
    ax.set_title(f"Developmental stage composition is stable across base changes\n{species}")
    ax.set_xlabel("Pyrimidine-normalized base change")
    ax.set_ylabel("Mean donor-level stage composition (%)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title="Stage", fontsize=11, title_fontsize=12, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.tight_layout()
    plt.show()
_print_donor_count("Mouse", base_change_donor_counts["Mouse"])
_print_donor_count("Human", base_change_donor_counts["Human"])
for species in ["Mouse", "Human"]:
    species_summary = (
        summary[summary["species"] == species]
        .pivot(index="mut_class", columns="stage_label", values="variant_percent")
        .reindex(index=MUT_TYPES, columns=STAGE_ORDER, fill_value=0.0)
        .round(2)
    )
    print(f"\n## {species} mean stage composition by base change")
    print(species_summary.to_string())


## Extended Data Fig. 8 - autosomal donor-variant count distributions


In [ ]:
CHROM_DIST_SPECS = [
    {
        "species": "Mouse",
        "title": "Mouse (TabMur)",
        "mut_csv": REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
        "fai": REPO_ROOT / "data" / "ref_genome" / "mm10.fa.fai",
        "chromosomes": [f"chr{i}" for i in range(1, 20)],
        "color": "#4C78A8",
        "bin_size_bp": 10_000_000,
    },
    {
        "species": "Human",
        "title": "Human (TabSap)",
        "mut_csv": REPO_ROOT / "data" / "TabSap" / "aggregates" / "mut_table_with_stage.csv",
        "fai": REPO_ROOT / "data" / "ref_genome" / "gencode_v41_ercc.fa.fai",
        "chromosomes": [f"chr{i}" for i in range(1, 23)],
        "color": "#54A24B",
        "bin_size_bp": 10_000_000,
    },
]


def _sem_or_zero(values):
    return float(values.sem(ddof=1)) if len(values) > 1 else 0.0


def _load_autosomal_donor_chrom_counts(mut_csv, fai_path, chromosomes):
    donors = sorted(
        {
            str(donor)
            for donor in pd.read_csv(
                mut_csv.parent / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
                index_col=0,
            ).index
        }
    )
    mut = pd.read_csv(mut_csv, usecols=["donor", "var_id", "#CHROM"], low_memory=False)
    mut["donor"] = mut["donor"].astype(str)
    mut = mut[mut["donor"].isin(donors)].dropna(subset=["var_id", "#CHROM"]).copy()
    mut["var_id"] = mut["var_id"].astype(str)
    mut["#CHROM"] = mut["#CHROM"].astype(str).str.strip()
    mut = mut[mut["#CHROM"].isin(chromosomes)].drop_duplicates(subset=["donor", "var_id"])

    lengths = pd.read_csv(
        fai_path,
        sep="\t",
        header=None,
        names=["chromosome", "length_bp"],
        usecols=[0, 1],
    )
    lengths = (
        lengths[lengths["chromosome"].isin(chromosomes)]
        .set_index("chromosome")
        .reindex(chromosomes)
    )

    donor_counts = (
        mut.groupby(["donor", "#CHROM"], observed=True)["var_id"]
        .nunique()
        .rename("n_variants")
        .reset_index()
        .rename(columns={"#CHROM": "chromosome"})
    )
    donor_grid = pd.MultiIndex.from_product(
        [donors, chromosomes],
        names=["donor", "chromosome"],
    ).to_frame(index=False)
    donor_counts = donor_grid.merge(donor_counts, on=["donor", "chromosome"], how="left")
    donor_counts["n_variants"] = donor_counts["n_variants"].fillna(0).astype(float)
    donor_counts = donor_counts.merge(lengths.reset_index(), on="chromosome", how="left")
    donor_counts["variants_per_100mb"] = donor_counts["n_variants"] / (
        donor_counts["length_bp"] / 100_000_000.0
    )
    donor_counts["chromosome"] = pd.Categorical(
        donor_counts["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    donor_counts = donor_counts.sort_values(["chromosome", "donor"]).reset_index(drop=True)

    chrom_summary = (
        donor_counts.groupby("chromosome", observed=True)
        .agg(
            mean_n_variants=("n_variants", "mean"),
            sem_n_variants=("n_variants", _sem_or_zero),
            mean_variants_per_100mb=("variants_per_100mb", "mean"),
            sem_variants_per_100mb=("variants_per_100mb", _sem_or_zero),
        )
        .reset_index()
    )
    chrom_summary["chromosome"] = pd.Categorical(
        chrom_summary["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    chrom_summary = chrom_summary.sort_values("chromosome").reset_index(drop=True)
    chrom_summary["n_donors"] = len(donors)

    donor_counts["chromosome"] = donor_counts["chromosome"].astype(str)
    chrom_summary["chromosome"] = chrom_summary["chromosome"].astype(str)
    return donors, donor_counts, chrom_summary, lengths


def _build_autosomal_bins(lengths_df, chromosomes, bin_size_bp):
    rows = []
    for chromosome in chromosomes:
        chrom_length = int(lengths_df.loc[chromosome, "length_bp"])
        n_bins = int(np.ceil(chrom_length / bin_size_bp))
        for bin_idx in range(n_bins):
            start_bp = bin_idx * bin_size_bp + 1
            end_bp = min((bin_idx + 1) * bin_size_bp, chrom_length)
            rows.append(
                {
                    "chromosome": chromosome,
                    "bin_idx": bin_idx,
                    "start_bp": start_bp,
                    "end_bp": end_bp,
                    "bin_width_bp": end_bp - start_bp + 1,
                }
            )
    bins = pd.DataFrame(rows)
    bins["chromosome"] = pd.Categorical(
        bins["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    return bins.sort_values(["chromosome", "bin_idx"]).reset_index(drop=True)


def _load_autosomal_donor_bin_counts(mut_csv, fai_path, chromosomes, bin_size_bp):
    donors, _, _, lengths = _load_autosomal_donor_chrom_counts(mut_csv, fai_path, chromosomes)
    bins = _build_autosomal_bins(lengths, chromosomes, bin_size_bp)

    mut = pd.read_csv(
        mut_csv,
        usecols=["donor", "var_id", "#CHROM", "Start"],
        low_memory=False,
    )
    mut["donor"] = mut["donor"].astype(str)
    mut = mut[
        mut["donor"].isin(donors)
        & mut["#CHROM"].astype(str).str.strip().isin(chromosomes)
    ].dropna(subset=["donor", "var_id", "#CHROM", "Start"]).copy()
    mut["var_id"] = mut["var_id"].astype(str)
    mut["#CHROM"] = mut["#CHROM"].astype(str).str.strip()
    mut["Start"] = pd.to_numeric(mut["Start"], errors="coerce")
    mut = mut.dropna(subset=["Start"]).drop_duplicates(subset=["donor", "var_id"])
    mut["Start"] = mut["Start"].astype(int)
    mut["bin_idx"] = ((mut["Start"] - 1) // bin_size_bp).astype(int)

    donor_bin_counts = (
        mut.groupby(["donor", "#CHROM", "bin_idx"], observed=True)["var_id"]
        .nunique()
        .rename("n_variants")
        .reset_index()
        .rename(columns={"#CHROM": "chromosome"})
    )
    donor_grid = bins[["chromosome", "bin_idx"]].drop_duplicates().copy()
    donor_grid["key"] = 1
    donor_grid = donor_grid.merge(
        pd.DataFrame({"donor": donors, "key": 1}),
        on="key",
        how="inner",
    ).drop(columns="key")
    donor_bin_counts = donor_grid.merge(
        donor_bin_counts,
        on=["donor", "chromosome", "bin_idx"],
        how="left",
    )
    donor_bin_counts["n_variants"] = donor_bin_counts["n_variants"].fillna(0).astype(float)
    donor_bin_counts = donor_bin_counts.merge(
        bins,
        on=["chromosome", "bin_idx"],
        how="left",
    )
    chrom_lengths = (
        bins.groupby("chromosome", observed=True)["bin_width_bp"].sum().rename("chrom_length_bp")
    )
    offsets = chrom_lengths.reset_index().copy()
    offsets["chromosome"] = pd.Categorical(
        offsets["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    offsets = offsets.sort_values("chromosome").reset_index(drop=True)
    offsets["chrom_offset_bp"] = offsets["chrom_length_bp"].cumsum().shift(fill_value=0).astype(float)
    offsets["chromosome"] = offsets["chromosome"].astype(str)

    donor_bin_counts["variants_per_100mb"] = donor_bin_counts["n_variants"] / (
        donor_bin_counts["bin_width_bp"] / 100_000_000.0
    )
    donor_bin_counts["bin_mid_bp"] = (
        donor_bin_counts["start_bp"] + donor_bin_counts["end_bp"]
    ) / 2.0
    donor_bin_counts = donor_bin_counts.merge(
        offsets[["chromosome", "chrom_length_bp", "chrom_offset_bp"]],
        on="chromosome",
        how="left",
    )
    donor_bin_counts["genome_mid_bp"] = donor_bin_counts["chrom_offset_bp"] + donor_bin_counts["bin_mid_bp"]

    bin_summary = (
        donor_bin_counts.groupby(
            ["chromosome", "bin_idx", "start_bp", "end_bp", "bin_width_bp", "bin_mid_bp"],
            observed=True,
        )
        .agg(
            mean_n_variants=("n_variants", "mean"),
            sem_n_variants=("n_variants", _sem_or_zero),
            mean_variants_per_100mb=("variants_per_100mb", "mean"),
            sem_variants_per_100mb=("variants_per_100mb", _sem_or_zero),
        )
        .reset_index()
    )
    bin_summary = bin_summary.merge(
        chrom_lengths.reset_index(),
        on="chromosome",
        how="left",
    )
    bin_summary["chromosome"] = pd.Categorical(
        bin_summary["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    bin_summary = bin_summary.sort_values(["chromosome", "bin_idx"]).reset_index(drop=True)
    bin_summary["chromosome_order"] = bin_summary["chromosome"].cat.codes
    offsets = offsets.merge(
        bin_summary[["chromosome", "chromosome_order"]].drop_duplicates(),
        on="chromosome",
        how="left",
    ).sort_values("chromosome_order")
    bin_summary = bin_summary.merge(
        offsets[["chromosome", "chrom_offset_bp"]],
        on="chromosome",
        how="left",
    )
    bin_summary["genome_mid_bp"] = bin_summary["chrom_offset_bp"] + bin_summary["bin_mid_bp"]
    bin_summary["start_mb"] = bin_summary["start_bp"] / 1_000_000.0
    bin_summary["end_mb"] = bin_summary["end_bp"] / 1_000_000.0
    bin_summary["region"] = (
        bin_summary["chromosome"].astype(str)
        + ":"
        + bin_summary["start_mb"].map(lambda value: f"{value:.1f}")
        + "-"
        + bin_summary["end_mb"].map(lambda value: f"{value:.1f}")
        + " Mb"
    )
    bin_summary["n_donors"] = len(donors)

    donor_bin_counts["chromosome"] = donor_bin_counts["chromosome"].astype(str)
    bin_summary["chromosome"] = bin_summary["chromosome"].astype(str)
    return donor_bin_counts, bin_summary, offsets


def _load_autosomal_variant_positions(mut_csv, fai_path, chromosomes):
    donors, _, _, lengths = _load_autosomal_donor_chrom_counts(mut_csv, fai_path, chromosomes)
    mut = pd.read_csv(
        mut_csv,
        usecols=["donor", "var_id", "#CHROM", "Start", "stage_label"],
        low_memory=False,
    )
    mut["donor"] = mut["donor"].astype(str)
    mut = mut[
        mut["donor"].isin(donors)
        & mut["#CHROM"].astype(str).str.strip().isin(chromosomes)
    ].dropna(subset=["donor", "var_id", "#CHROM", "Start", "stage_label"]).copy()
    mut["var_id"] = mut["var_id"].astype(str)
    mut["stage_label"] = mut["stage_label"].astype(str).str.strip()
    mut = mut[mut["stage_label"].isin(STAGE_ORDER)].copy()
    mut["chromosome"] = mut["#CHROM"].astype(str).str.strip()
    mut["Start"] = pd.to_numeric(mut["Start"], errors="coerce")
    mut = mut.dropna(subset=["Start"]).drop_duplicates(subset=["donor", "var_id"])
    mut["Start"] = mut["Start"].astype(int)
    mut["stage_label"] = pd.Categorical(
        mut["stage_label"],
        categories=STAGE_ORDER,
        ordered=True,
    )
    mut["chromosome"] = pd.Categorical(
        mut["chromosome"],
        categories=chromosomes,
        ordered=True,
    )
    mut = mut.sort_values(["stage_label", "chromosome", "Start", "donor", "var_id"]).reset_index(drop=True)
    mut = mut.merge(
        lengths.reset_index().rename(columns={"index": "chromosome"}),
        on="chromosome",
        how="left",
    )
    mut["stage_label"] = mut["stage_label"].astype(str)
    mut["chromosome"] = mut["chromosome"].astype(str)
    mut["chromosome_order"] = pd.Categorical(
        mut["chromosome"],
        categories=chromosomes,
        ordered=True,
    ).codes
    mut["position_mb"] = mut["Start"] / 1_000_000.0
    mut["position_fraction"] = mut["Start"] / mut["length_bp"]
    mut["position_percent"] = mut["position_fraction"] * 100.0
    return donors, mut, lengths


def _plot_chromosome_barplot(ax, summary_df, mean_col, sem_col, ylabel, title, color):
    x = np.arange(len(summary_df))
    ax.bar(
        x,
        summary_df[mean_col].to_numpy(float),
        yerr=summary_df[sem_col].fillna(0).to_numpy(float),
        capsize=3.5,
        color=color,
        alpha=0.82,
        edgecolor="black",
        linewidth=0.5,
        error_kw={"elinewidth": 1.0, "ecolor": "black"},
        zorder=2,
    )
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Autosomal chromosome")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["chromosome"].tolist(), rotation=90)
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _plot_chromosome_position_strip(
    ax,
    variant_df,
    chromosomes,
    title,
    color,
    seed=0,
):
    rng = np.random.default_rng(seed)
    x_base = variant_df["chromosome_order"].to_numpy(float)
    x_jitter = rng.uniform(-0.28, 0.28, size=len(variant_df))
    y = variant_df["position_percent"].to_numpy(float)

    ax.scatter(
        x_base + x_jitter,
        y,
        s=8,
        color=color,
        alpha=0.18,
        edgecolor="none",
        rasterized=True,
        zorder=2,
    )

    centerline = (
        variant_df.groupby("chromosome", observed=True)["position_percent"]
        .median()
        .reindex(chromosomes)
    )
    x_line = np.arange(len(chromosomes), dtype=float)
    ax.plot(
        x_line,
        centerline.to_numpy(float),
        color="black",
        linewidth=1.7,
        zorder=4,
    )
    ax.scatter(
        x_line,
        centerline.to_numpy(float),
        s=20,
        facecolor="white",
        edgecolor="black",
        linewidth=0.6,
        zorder=5,
    )

    for x in np.arange(-0.5, len(chromosomes), 1.0):
        ax.axvline(x, color="#D9D9D9", linewidth=0.45, alpha=0.6, zorder=1)
    ax.axhline(50.0, color="#8C8C8C", linestyle="--", linewidth=0.8, alpha=0.8, zorder=1)

    ax.set_title(title)
    ax.set_xlabel("Autosomal chromosome")
    ax.set_ylabel("Mutation position within chromosome (%)")
    ax.set_xticks(np.arange(len(chromosomes)))
    ax.set_xticklabels([chrom.replace("chr", "") for chrom in chromosomes], rotation=90)
    ax.set_xlim(-0.6, len(chromosomes) - 0.4)
    ax.set_ylim(0, 100)
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _add_genome_guides(ax, offsets_df):
    offsets_df = offsets_df.sort_values("chrom_offset_bp").reset_index(drop=True)
    for idx, row in offsets_df.iterrows():
        start = float(row["chrom_offset_bp"]) / 1_000_000.0
        end = start + float(row["chrom_length_bp"]) / 1_000_000.0
        if idx % 2 == 0:
            ax.axvspan(start, end, color="#000000", alpha=0.04, lw=0, zorder=0)
        ax.axvline(start, color="#BBBBBB", linewidth=0.5, alpha=0.8, zorder=1)
    last_row = offsets_df.iloc[-1]
    ax.axvline(
        (float(last_row["chrom_offset_bp"]) + float(last_row["chrom_length_bp"])) / 1_000_000.0,
        color="#BBBBBB",
        linewidth=0.5,
        alpha=0.8,
        zorder=1,
    )


def _set_genome_axis(ax, offsets_df, x_values_mb):
    centers = (
        offsets_df["chrom_offset_bp"].to_numpy(float)
        + offsets_df["chrom_length_bp"].to_numpy(float) / 2.0
    ) / 1_000_000.0
    labels = [chrom.replace("chr", "") for chrom in offsets_df["chromosome"].tolist()]
    ax.set_xticks(centers)
    ax.set_xticklabels(labels, rotation=90)
    ax.set_xlim(float(np.min(x_values_mb)) - 5, float(np.max(x_values_mb)) + 5)
    ax.set_xlabel("Autosomal genomic position (chromosomes in order)")
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _plot_genomewide_mean_density(ax, summary_df, offsets_df, title, color):
    _add_genome_guides(ax, offsets_df)
    x = summary_df["genome_mid_bp"].to_numpy(float) / 1_000_000.0
    y = summary_df["mean_variants_per_100mb"].to_numpy(float)
    sem = summary_df["sem_variants_per_100mb"].fillna(0).to_numpy(float)
    ax.fill_between(
        x,
        np.clip(y - sem, a_min=0, a_max=None),
        y + sem,
        color=color,
        alpha=0.18,
        linewidth=0,
        zorder=2,
    )
    ax.plot(x, y, color=color, linewidth=1.4, zorder=3)
    ax.scatter(x, y, s=12, color=color, alpha=0.6, edgecolor="white", linewidth=0.2, zorder=4)

    top_bins = summary_df.nlargest(8, "mean_variants_per_100mb").copy()
    ax.scatter(
        top_bins["genome_mid_bp"].to_numpy(float) / 1_000_000.0,
        top_bins["mean_variants_per_100mb"].to_numpy(float),
        s=24,
        color="#C44E52",
        edgecolor="black",
        linewidth=0.35,
        zorder=5,
    )

    ax.set_title(title)
    ax.set_ylabel("Mean unique donor-variants per donor per 100 Mb")
    _set_genome_axis(ax, offsets_df, x)


def _plot_genomewide_donor_facets(donor_df, offsets_df, title, color):
    donors = donor_df["donor"].drop_duplicates().tolist()
    n_donors = len(donors)
    ncols = 2 if n_donors <= 8 else 3
    nrows = int(np.ceil(n_donors / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(max(14, ncols * 6), max(2.8 * nrows, 4.8)),
        dpi=900,
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, donor in zip(axes, donors):
        sub = donor_df[donor_df["donor"] == donor].sort_values(["chromosome", "bin_idx"]).reset_index(drop=True)
        _add_genome_guides(ax, offsets_df)
        for _, chrom_sub in sub.groupby("chromosome", sort=False):
            x = chrom_sub["genome_mid_bp"].to_numpy(float) / 1_000_000.0
            y = chrom_sub["variants_per_100mb"].to_numpy(float)
            ax.plot(x, y, color=color, linewidth=0.9, alpha=0.85, zorder=3)
            ax.scatter(x, y, s=7, color=color, alpha=0.45, edgecolor="none", zorder=4)
        ax.set_title(str(donor), fontsize=9)
        x = sub["genome_mid_bp"].to_numpy(float) / 1_000_000.0
        _set_genome_axis(ax, offsets_df, x)
    for ax in axes[n_donors:]:
        ax.set_visible(False)
    for ax in axes[::ncols]:
        ax.set_ylabel("Unique donor-variants per 100 Mb")
    fig.suptitle(title, y=1.01)
    plt.show()


autosomal_chrom_summary_tables = []
autosomal_donor_chrom_tables = []
autosomal_bin_summary_tables = []
autosomal_donor_bin_tables = []
autosomal_top_region_tables = []
autosomal_variant_position_tables = []

for spec in CHROM_DIST_SPECS:
    donors, donor_df, chrom_summary, lengths = _load_autosomal_donor_chrom_counts(
        spec["mut_csv"],
        spec["fai"],
        spec["chromosomes"],
    )
    _, variant_positions, _ = _load_autosomal_variant_positions(
        spec["mut_csv"],
        spec["fai"],
        spec["chromosomes"],
    )
    donor_bin_df, bin_summary, offsets = _load_autosomal_donor_bin_counts(
        spec["mut_csv"],
        spec["fai"],
        spec["chromosomes"],
        spec["bin_size_bp"],
    )

    autosomal_donor_chrom_tables.append(
        donor_df.assign(species=spec["species"], title=spec["title"])[
            [
                "species",
                "title",
                "donor",
                "chromosome",
                "n_variants",
                "variants_per_100mb",
                "length_bp",
            ]
        ]
    )
    autosomal_chrom_summary_tables.append(
        chrom_summary.assign(species=spec["species"], title=spec["title"], color=spec["color"])[
            [
                "species",
                "title",
                "chromosome",
                "color",
                "n_donors",
                "mean_n_variants",
                "sem_n_variants",
                "mean_variants_per_100mb",
                "sem_variants_per_100mb",
            ]
        ]
    )
    autosomal_donor_bin_tables.append(
        donor_bin_df.assign(species=spec["species"], title=spec["title"])[
            [
                "species",
                "title",
                "donor",
                "chromosome",
                "bin_idx",
                "start_bp",
                "end_bp",
                "bin_width_bp",
                "n_variants",
                "variants_per_100mb",
                "genome_mid_bp",
            ]
        ]
    )
    autosomal_bin_summary_tables.append(
        bin_summary.assign(species=spec["species"], title=spec["title"], color=spec["color"])[
            [
                "species",
                "title",
                "chromosome",
                "bin_idx",
                "start_bp",
                "end_bp",
                "start_mb",
                "end_mb",
                "region",
                "bin_width_bp",
                "n_donors",
                "mean_n_variants",
                "sem_n_variants",
                "mean_variants_per_100mb",
                "sem_variants_per_100mb",
                "genome_mid_bp",
            ]
        ]
    )
    autosomal_variant_position_tables.append(
        variant_positions.assign(species=spec["species"], title=spec["title"])[
            [
                "species",
                "title",
                "donor",
                "var_id",
                "stage_label",
                "chromosome",
                "chromosome_order",
                "Start",
                "length_bp",
                "position_mb",
                "position_fraction",
                "position_percent",
            ]
        ]
    )

    top_regions = (
        bin_summary.nlargest(20, "mean_variants_per_100mb")
        .reset_index(drop=True)
        .copy()
    )
    top_regions.insert(0, "rank", np.arange(1, len(top_regions) + 1))
    autosomal_top_region_tables.append(
        top_regions.assign(
            species=spec["species"],
            title=spec["title"],
            bin_size_mb=spec["bin_size_bp"] / 1_000_000.0,
        )[
            [
                "species",
                "title",
                "rank",
                "chromosome",
                "region",
                "bin_size_mb",
                "mean_n_variants",
                "sem_n_variants",
                "mean_variants_per_100mb",
                "sem_variants_per_100mb",
                "n_donors",
            ]
        ]
    )

    sub = chrom_summary.copy().reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, 5), dpi=900, constrained_layout=True)
    _plot_chromosome_barplot(
        ax,
        sub,
        mean_col="mean_variants_per_100mb",
        sem_col="sem_variants_per_100mb",
        ylabel="Unique donor-variants per donor per 100 Mb",
        title=f"Autosomal donor-variant burden by chromosome\n{spec['title']} size-normalized chromosome means (+/- SEM)",
        color=spec["color"],
    )
    plt.show()

    for stage_idx, stage in enumerate(STAGE_ORDER):
        stage_variant_positions = variant_positions[variant_positions["stage_label"] == stage].copy()
        fig, ax = plt.subplots(figsize=(11.5, 5.2), dpi=900, constrained_layout=True)
        _plot_chromosome_position_strip(
            ax,
            stage_variant_positions,
            spec["chromosomes"],
            title=(
                f"{spec['title']} {stage} autosomal mutation positions by chromosome\n"
                f"n={len(stage_variant_positions):,} unique donor-variants; black line connects the chromosome-wise median position"
            ),
            color=spec["color"],
            seed=1729 + stage_idx,
        )
        plt.show()

    fig, ax = plt.subplots(figsize=(14, 4.8), dpi=900, constrained_layout=True)
    _plot_genomewide_mean_density(
        ax,
        bin_summary,
        offsets,
        title=(
            f"{spec['title']} autosomal mutation density across genomic position\n"
            f"{int(spec['bin_size_bp'] / 1_000_000)} Mb bins; points are donor means and shading is +/- SEM"
        ),
        color=spec["color"],
    )
    plt.show()

    _plot_genomewide_donor_facets(
        donor_bin_df,
        offsets,
        title=(
            f"{spec['title']} donor-level autosomal mutation density across genomic position\n"
            f"{int(spec['bin_size_bp'] / 1_000_000)} Mb bins; one panel per donor"
        ),
        color=spec["color"],
    )

    _print_donor_count(spec["title"], len(donors))
    print(
        f"\n{spec['title']} highest-density autosomal regions "
        f"({int(spec['bin_size_bp'] / 1_000_000)} Mb bins)"
    )
    print(
        autosomal_top_region_tables[-1][
            [
                "rank",
                "chromosome",
                "region",
                "mean_n_variants",
                "sem_n_variants",
                "mean_variants_per_100mb",
                "sem_variants_per_100mb",
            ]
        ].round(
            {
                "mean_n_variants": 2,
                "sem_n_variants": 2,
                "mean_variants_per_100mb": 2,
                "sem_variants_per_100mb": 2,
            }
        ).to_string(index=False)
    )

autosomal_chrom_summary = pd.concat(autosomal_chrom_summary_tables, ignore_index=True)
autosomal_donor_chrom_counts = pd.concat(autosomal_donor_chrom_tables, ignore_index=True)
autosomal_bin_summary = pd.concat(autosomal_bin_summary_tables, ignore_index=True)
autosomal_donor_bin_counts = pd.concat(autosomal_donor_bin_tables, ignore_index=True)
autosomal_top_regions = pd.concat(autosomal_top_region_tables, ignore_index=True)
autosomal_variant_positions = pd.concat(autosomal_variant_position_tables, ignore_index=True)

print(
    "\nAutosomes only. Whole-chromosome bars show size-normalized donor means (+/- SEM). "
    "The chromosome strip plots are split into developmental-stage panels, place every unique donor-variant within its chromosome, and connect the per-chromosome median position with a line from the first to the last autosome. "
    "Genome-wide regional plots show autosomal 10 Mb bins ordered along chromosomes. "
    "The cohort plot shows the observed donor mean in each bin with a SEM band, "
    "and the donor figure shows the unsmoothed per-donor profiles in separate panels. "
    "The highest-density bins are highlighted and listed below each plot."
)


## Extended Data Fig. 9 - autosomal position distributions by developmental stage


In [ ]:
                                                                                                           
                                                                                                                   
                                                                                
                                                                                                            
import pysam

CHROM_DIST_SPECS = [
    {
        "species": "Mouse",
        "title": "Mouse (TabMur)",
        "mut_csv": REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
        "fai": REPO_ROOT / "data" / "ref_genome" / "mm10.fa.fai",
        "chromosomes": [f"chr{i}" for i in range(1, 20)],
        "color": "#4C78A8",
    },
    {
        "species": "Human",
        "title": "Human (TabSap)",
        "mut_csv": REPO_ROOT / "data" / "TabSap" / "aggregates" / "mut_table_with_stage.csv",
        "fai": REPO_ROOT / "data" / "ref_genome" / "gencode_v41_ercc.fa.fai",
        "chromosomes": [f"chr{i}" for i in range(1, 23)],
        "color": "#54A24B",
    },
]


def _sem_or_zero(values):
    return float(values.sem(ddof=1)) if len(values) > 1 else 0.0


def _load_autosomal_donor_chrom_counts(mut_csv, fai_path, chromosomes):
    donors = sorted(
        {
            str(donor)
            for donor in pd.read_csv(
                mut_csv.parent / "stage_burden_per_donor_perkb_STANDARDIZED_3LEVEL_cells.csv",
                index_col=0,
            ).index
        }
    )
    lengths = pd.read_csv(
        fai_path,
        sep="\t",
        header=None,
        names=["chromosome", "length_bp"],
        usecols=[0, 1],
    )
    lengths = (
        lengths[lengths["chromosome"].isin(chromosomes)]
        .set_index("chromosome")
        .reindex(chromosomes)
    )
    donor_counts = pd.DataFrame()
    chrom_summary = pd.DataFrame()
    return donors, donor_counts, chrom_summary, lengths

TEL_END_BIN_EDGES_MB = np.array([0, 5, 10, 20, 40, 80, 160], dtype=float)
TEL_END_BIN_LABELS = [
    f"{int(left)}-{int(right)} Mb"
    for left, right in zip(TEL_END_BIN_EDGES_MB[:-1], TEL_END_BIN_EDGES_MB[1:])
]
SBS_CLASSES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
SBS96_FLANK_ORDER = [left + right for left in "ACGT" for right in "ACGT"]
SBS96_ORDER = [
    f"{left}[{sub}]{right}"
    for sub in SBS_CLASSES
    for left in "ACGT"
    for right in "ACGT"
]
SBS_CLASS_COLORS = {
    "C>A": "#4E79A7",
    "C>G": "#E15759",
    "C>T": "#76B7B2",
    "T>A": "#F28E2B",
    "T>C": "#59A14F",
    "T>G": "#B07AA1",
}
_CONTEXT_COMPLEMENT = str.maketrans("ACGT", "TGCA")
CONTEXT_FASTA_CANDIDATES = {
    "Mouse": [REPO_ROOT / "data" / "ref_genome" / "mm10.fa", _env_path("LORE2026_MM10_FASTA")],
    "Human": [REPO_ROOT / "data" / "ref_genome" / "gencode_v41_ercc.fa"],
}


def _resolve_first_existing_path(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return None


def _load_autosomal_unique_donor_variants(spec, extra_cols=None):
    extra_cols = extra_cols or []
    donors, _, _, lengths = _load_autosomal_donor_chrom_counts(
        spec["mut_csv"],
        spec["fai"],
        spec["chromosomes"],
    )
    usecols = ["donor", "var_id", "#CHROM", "Start"] + extra_cols
    mut = pd.read_csv(spec["mut_csv"], usecols=usecols, low_memory=False)
    mut["donor"] = mut["donor"].astype(str)
    mut["chromosome"] = mut["#CHROM"].astype(str).str.strip()
    mut = mut[
        mut["donor"].isin(donors)
        & mut["chromosome"].isin(spec["chromosomes"])
    ].dropna(subset=["var_id", "Start"]).copy()
    mut["Start"] = pd.to_numeric(mut["Start"], errors="coerce")
    mut = mut.dropna(subset=["Start"]).drop_duplicates(subset=["donor", "var_id"])
    mut["Start"] = mut["Start"].astype(int)
    mut = mut.merge(lengths.reset_index(), on="chromosome", how="left")
    return donors, mut.reset_index(drop=True), lengths


def _build_telomere_territory_table(lengths, edges_mb):
    rows = []
    edges_bp = edges_mb * 1_000_000.0
    for left_bp, right_bp, label in zip(edges_bp[:-1], edges_bp[1:], TEL_END_BIN_LABELS):
        territory_bp = 0.0
        for _, row in lengths.iterrows():
            half_chrom_bp = float(row["length_bp"]) / 2.0
            overlap_bp = max(0.0, min(half_chrom_bp, right_bp) - left_bp)
            territory_bp += 2.0 * overlap_bp
        rows.append({"telomere_bin": label, "territory_bp": territory_bp})
    territory = pd.DataFrame(rows)
    territory["territory_fraction"] = territory["territory_bp"] / territory["territory_bp"].sum()
    territory["telomere_bin"] = pd.Categorical(
        territory["telomere_bin"],
        categories=TEL_END_BIN_LABELS,
        ordered=True,
    )
    return territory.sort_values("telomere_bin").reset_index(drop=True)


def _build_telomere_enrichment_tables(spec):
    donors, mut, lengths = _load_autosomal_unique_donor_variants(spec)
    mut["telomere_dist_mb"] = np.minimum(mut["Start"] - 1, mut["length_bp"] - mut["Start"]) / 1_000_000.0
    mut["telomere_bin"] = pd.cut(
        mut["telomere_dist_mb"],
        bins=TEL_END_BIN_EDGES_MB,
        labels=TEL_END_BIN_LABELS,
        include_lowest=True,
        right=False,
    )
    territory = _build_telomere_territory_table(lengths, TEL_END_BIN_EDGES_MB)
    donor_totals = (
        mut.groupby("donor", observed=True)["var_id"]
        .nunique()
        .rename("donor_total")
        .reset_index()
    )
    donor_counts = (
        mut.groupby(["donor", "telomere_bin"], observed=True)["var_id"]
        .nunique()
        .rename("n_variants")
        .reset_index()
    )
    donor_grid = pd.MultiIndex.from_product(
        [donors, TEL_END_BIN_LABELS],
        names=["donor", "telomere_bin"],
    ).to_frame(index=False)
    donor_counts = donor_grid.merge(donor_counts, on=["donor", "telomere_bin"], how="left")
    donor_counts["n_variants"] = donor_counts["n_variants"].fillna(0).astype(float)
    donor_counts["telomere_bin"] = pd.Categorical(
        donor_counts["telomere_bin"],
        categories=TEL_END_BIN_LABELS,
        ordered=True,
    )
    donor_counts = donor_counts.merge(donor_totals, on="donor", how="left")
    donor_counts = donor_counts.merge(territory, on="telomere_bin", how="left")
    donor_counts["observed_fraction"] = donor_counts["n_variants"] / donor_counts["donor_total"]
    donor_counts["fold_enrichment"] = donor_counts["observed_fraction"] / donor_counts["territory_fraction"]
    donor_counts["variants_per_gb"] = donor_counts["n_variants"] / (donor_counts["territory_bp"] / 1_000_000_000.0)

    summary = (
        donor_counts.groupby("telomere_bin", observed=True)
        .agg(
            mean_fold_enrichment=("fold_enrichment", "mean"),
            sem_fold_enrichment=("fold_enrichment", _sem_or_zero),
            mean_variants_per_gb=("variants_per_gb", "mean"),
            sem_variants_per_gb=("variants_per_gb", _sem_or_zero),
        )
        .reset_index()
        .sort_values("telomere_bin")
        .reset_index(drop=True)
    )
    summary["n_donors"] = len(donors)

    donor_counts["telomere_bin"] = donor_counts["telomere_bin"].astype(str)
    summary["telomere_bin"] = summary["telomere_bin"].astype(str)
    return donor_counts, summary


def _plot_telomere_enrichment(ax, summary_df, title, color):
    x = np.arange(len(summary_df))
    ax.bar(
        x,
        summary_df["mean_fold_enrichment"].to_numpy(float),
        yerr=summary_df["sem_fold_enrichment"].fillna(0).to_numpy(float),
        capsize=3.5,
        color=color,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.5,
        error_kw={"elinewidth": 1.0, "ecolor": "black"},
        zorder=2,
    )
    ax.axhline(1.0, color="#555555", linestyle="--", linewidth=0.9, alpha=0.8, zorder=1)
    ax.set_title(title)
    ax.set_xlabel("Distance to nearest chromosome end")
    ax.set_ylabel("Fold enrichment vs reference territory")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["telomere_bin"].tolist(), rotation=30, ha="right")
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _reverse_complement(seq):
    return str(seq).translate(_CONTEXT_COMPLEMENT)[::-1]


def _canonical_sbs96_context(ref, alt, context):
    ref = str(ref).upper()
    alt = str(alt).upper()
    context = str(context).upper()
    if len(context) != 3 or any(base not in "ACGT" for base in context):
        return None
    if ref not in "ACGT" or alt not in "ACGT" or ref == alt:
        return None
    if context[1] != ref:
        return None
    if ref in {"A", "G"}:
        ref = ref.translate(_CONTEXT_COMPLEMENT)
        alt = alt.translate(_CONTEXT_COMPLEMENT)
        context = _reverse_complement(context)
    if ref not in {"C", "T"}:
        return None
    return f"{context[0]}[{ref}>{alt}]{context[2]}"


def _build_sbs96_tables(spec):
    fasta_path = _resolve_first_existing_path(CONTEXT_FASTA_CANDIDATES.get(spec["species"], []))
    if fasta_path is None:
        return None, None, None

    _, mut, _ = _load_autosomal_unique_donor_variants(spec, extra_cols=["REF", "ALT_expected"])
    fa = pysam.FastaFile(str(fasta_path))
    contexts = []
    skipped = 0
    for row in mut.itertuples(index=False):
        pos = int(row.Start)
        chrom_length = int(row.length_bp)
        if pos <= 1 or pos >= chrom_length:
            skipped += 1
            continue
        context = fa.fetch(str(row.chromosome), pos - 2, pos + 1).upper()
        if len(context) != 3:
            skipped += 1
            continue
        sbs96 = _canonical_sbs96_context(row.REF, row.ALT_expected, context)
        if sbs96 is None:
            skipped += 1
            continue
        contexts.append(sbs96)
    fa.close()

    sbs96 = (
        pd.Series(contexts, name="sbs96")
        .value_counts()
        .reindex(SBS96_ORDER, fill_value=0)
        .rename_axis("sbs96")
        .reset_index(name="n_variants")
    )
    sbs96["substitution"] = sbs96["sbs96"].str.extract(r"\[([^\]]+)\]")[0]
    sbs96["flanks"] = sbs96["sbs96"].str[0] + sbs96["sbs96"].str[-1]
    sbs96["fraction"] = sbs96["n_variants"] / sbs96["n_variants"].sum()
    sbs96["percent"] = sbs96["fraction"] * 100.0

    sbs6 = (
        sbs96.groupby("substitution", observed=True, as_index=False)["n_variants"]
        .sum()
    )
    sbs6["substitution"] = pd.Categorical(sbs6["substitution"], categories=SBS_CLASSES, ordered=True)
    sbs6 = sbs6.sort_values("substitution").reset_index(drop=True)
    sbs6["fraction"] = sbs6["n_variants"] / sbs6["n_variants"].sum()
    sbs6["percent"] = sbs6["fraction"] * 100.0

    meta = {
        "species": spec["species"],
        "title": spec["title"],
        "fasta_path": str(fasta_path),
        "n_total_variants": int(len(mut)),
        "n_context_annotated": int(len(contexts)),
        "n_skipped": int(skipped),
    }
    return sbs96, sbs6, meta


def _plot_sbs96_heatmap(ax, sbs96_df, title, vmax):
    matrix = (
        sbs96_df.pivot(index="substitution", columns="flanks", values="percent")
        .reindex(index=SBS_CLASSES, columns=SBS96_FLANK_ORDER)
        .fillna(0.0)
    )
    sns.heatmap(
        matrix,
        ax=ax,
        cmap="mako",
        vmin=0.0,
        vmax=vmax,
        cbar_kws={"label": "% of unique donor-variants"},
    )
    ax.set_title(title)
    ax.set_xlabel("Flanking bases (5' then 3')")
    ax.set_ylabel("Pyrimidine-centered substitution")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)


def _plot_sbs6_bar(ax, sbs6_df, title):
    colors = [SBS_CLASS_COLORS[str(sub)] for sub in sbs6_df["substitution"]]
    x = np.arange(len(sbs6_df))
    ax.bar(
        x,
        sbs6_df["percent"].to_numpy(float),
        color=colors,
        edgecolor="black",
        linewidth=0.5,
        alpha=0.9,
    )
    ax.set_title(title)
    ax.set_xlabel("Base substitution class")
    ax.set_ylabel("% of unique donor-variants")
    ax.set_xticks(x)
    ax.set_xticklabels(sbs6_df["substitution"].astype(str).tolist(), rotation=30, ha="right")
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


autosomal_telomere_bin_tables = []
autosomal_telomere_summary_tables = []
autosomal_sbs96_tables = []
autosomal_sbs6_tables = []
autosomal_sbs96_meta = []

for spec in CHROM_DIST_SPECS:
    telomere_counts, telomere_summary = _build_telomere_enrichment_tables(spec)
    autosomal_telomere_bin_tables.append(
        telomere_counts.assign(species=spec["species"], title=spec["title"])[
            [
                "species",
                "title",
                "donor",
                "telomere_bin",
                "n_variants",
                "donor_total",
                "territory_bp",
                "territory_fraction",
                "observed_fraction",
                "fold_enrichment",
                "variants_per_gb",
            ]
        ]
    )
    autosomal_telomere_summary_tables.append(
        telomere_summary.assign(species=spec["species"], title=spec["title"], color=spec["color"])[
            [
                "species",
                "title",
                "telomere_bin",
                "color",
                "n_donors",
                "mean_fold_enrichment",
                "sem_fold_enrichment",
                "mean_variants_per_gb",
                "sem_variants_per_gb",
            ]
        ]
    )

    sbs96, sbs6, sbs_meta = _build_sbs96_tables(spec)
    if sbs96 is not None:
        autosomal_sbs96_tables.append(
            sbs96.assign(species=spec["species"], title=spec["title"])[
                [
                    "species",
                    "title",
                    "sbs96",
                    "substitution",
                    "flanks",
                    "n_variants",
                    "fraction",
                    "percent",
                ]
            ]
        )
        autosomal_sbs6_tables.append(
            sbs6.assign(species=spec["species"], title=spec["title"])[
                [
                    "species",
                    "title",
                    "substitution",
                    "n_variants",
                    "fraction",
                    "percent",
                ]
            ]
        )
        autosomal_sbs96_meta.append(sbs_meta)

fig, axes = plt.subplots(1, len(CHROM_DIST_SPECS), figsize=(14, 4.8), dpi=900, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, spec, summary_df in zip(axes, CHROM_DIST_SPECS, autosomal_telomere_summary_tables):
    _plot_telomere_enrichment(
        ax,
        summary_df,
        title=(
            f"{spec['title']} mutation enrichment by telomere distance\n"
            "Unique donor-variants per donor normalized by reference territory"
        ),
        color=spec["color"],
    )
plt.show()

autosomal_telomere_bin_counts = pd.concat(autosomal_telomere_bin_tables, ignore_index=True)
autosomal_telomere_enrichment_summary = pd.concat(autosomal_telomere_summary_tables, ignore_index=True)

if autosomal_sbs96_tables:
    autosomal_sbs96_summary = pd.concat(autosomal_sbs96_tables, ignore_index=True)
    autosomal_sbs6_summary = pd.concat(autosomal_sbs6_tables, ignore_index=True)
    autosomal_sbs96_metadata = pd.DataFrame(autosomal_sbs96_meta)

    species_order = [spec["species"] for spec in CHROM_DIST_SPECS if spec["species"] in set(autosomal_sbs96_summary["species"])]
    vmax = float(autosomal_sbs96_summary["percent"].max())
    fig, axes = plt.subplots(
        len(species_order),
        2,
        figsize=(13.5, max(4.2 * len(species_order), 4.2)),
        dpi=900,
        constrained_layout=True,
        gridspec_kw={"width_ratios": [4.2, 1.3]},
    )
    axes = np.atleast_2d(axes)
    for row_idx, species in enumerate(species_order):
        spec = next(item for item in CHROM_DIST_SPECS if item["species"] == species)
        sbs96_df = autosomal_sbs96_summary[autosomal_sbs96_summary["species"] == species].copy()
        sbs6_df = autosomal_sbs6_summary[autosomal_sbs6_summary["species"] == species].copy()
        sbs6_df["substitution"] = pd.Categorical(sbs6_df["substitution"], categories=SBS_CLASSES, ordered=True)
        sbs6_df = sbs6_df.sort_values("substitution").reset_index(drop=True)
        _plot_sbs96_heatmap(
            axes[row_idx, 0],
            sbs96_df,
            title=f"{spec['title']} trinucleotide mutation spectrum (SBS96)",
            vmax=vmax,
        )
        _plot_sbs6_bar(
            axes[row_idx, 1],
            sbs6_df,
            title=f"{spec['title']} collapsed SBS6 profile",
        )
    plt.show()
else:
    autosomal_sbs96_summary = pd.DataFrame()
    autosomal_sbs6_summary = pd.DataFrame()
    autosomal_sbs96_metadata = pd.DataFrame()
    print("No chromosome FASTA was available, so the SBS96 context plots were skipped.")

print(
    "\nThese additional panels stay within the resources bundled here: chromosome lengths and FASTA sequence. "
    "They add a spatial summary relative to chromosome ends and a mechanistic summary of local sequence context. "
    "Gene-model context (coding/intronic/intergenic/promoter) and synonymous/missense/splice consequence panels would still require external gene annotations or a precomputed variant consequence table."
)
print("\nReference FASTA used for SBS96 context:")
if autosomal_sbs96_metadata.empty:
    print("None available")
else:
    print(autosomal_sbs96_metadata[["species", "fasta_path", "n_context_annotated", "n_total_variants", "n_skipped"]].to_string(index=False))
    print("\nTop trinucleotide contexts by species (% of unique donor-variants):")
    for species in autosomal_sbs96_metadata["species"]:
        top_contexts = (
            autosomal_sbs96_summary[autosomal_sbs96_summary["species"] == species]
            .nlargest(5, "percent")[["sbs96", "percent"]]
            .copy()
        )
        top_contexts["percent"] = top_contexts["percent"].round(2)
        print(f"\n{species}")
        print(top_contexts.to_string(index=False))



## Extended Data Fig. 10 - expression-mutation correlations and GO bars


In [ ]:
_plot_slope_volcano(TABMUR_DIR / "expression_coupling" / "slopes_protein_coding.csv", "Mice")


In [ ]:
_plot_slope_volcano(TABSAP_DIR / "expression_coupling" / "slopes_protein_coding.csv", "Humans")


In [ ]:
_plot_go_bar(TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_neg_1758807683509.tsv", TABMUR_DIR / "expression_coupling" / "gsea_report_for_na_pos_1758807683509.tsv", "Mouse GO Enrichment (Top 10 Pos & Neg)")


In [ ]:
_plot_go_bar(_latest_gsea_report(TABSAP_DIR / "expression_coupling", "neg", "TabSap"), _latest_gsea_report(TABSAP_DIR / "expression_coupling", "pos", "TabSap"), "Human GO Enrichment (Top 10 Pos & Neg)")


## Extended Data Fig. 11 - cell-type-controlled expression-mutation coupling


In [ ]:
def _control_base(species_label):
    if species_label == 'TabMur':
        return REPO_ROOT / 'data' / 'TabMur', 'Mouse (TabMur)', 'Mouse'
    return REPO_ROOT / 'data' / 'TabSap', 'Human (TabSap)', 'Human'


def _prepare_control_frame(df, global_col, control_col):
    out = df.copy()
    out['global_metric'] = pd.to_numeric(out[global_col], errors='coerce')
    out['control_metric'] = pd.to_numeric(out[control_col], errors='coerce')
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=['global_metric', 'control_metric']).copy()
    out['same_sign'] = np.sign(out['global_metric']) == np.sign(out['control_metric'])
    out['attenuation'] = (
        np.abs(out['control_metric'])
        .div(np.abs(out['global_metric']).replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
    )
    return out


def _load_celltype_control_comparison(species_label):
    base, title, _ = _control_base(species_label)
    global_path = base / 'expression_coupling' / 'slopes_protein_coding.csv'
    group_path = base / 'expression_coupling_by_group' / 'slopes_by_group_protein_coding.csv'
    global_df = pd.read_csv(global_path)
    key_col = 'gene_symbol' if ('gene_symbol' in global_df.columns and species_label == 'TabSap') else 'gene'
    global_df['gene_key'] = global_df[key_col].astype(str).str.upper()

    group_df = pd.read_csv(group_path)
    group_df['gene_key'] = group_df['gene'].astype(str).str.upper()
    group_df = group_df[group_df['n_cells_expr'] >= 100].copy()
    weighted = group_df.assign(weighted_slope=group_df['slope_z'] * group_df['n_cells_expr'])
    meta = (
        weighted.groupby('gene_key', observed=True)
        .agg(
            weighted_slope_sum=('weighted_slope', 'sum'),
            group_cells=('n_cells_expr', 'sum'),
        )
        .reset_index()
    )
    meta['within_slope'] = meta['weighted_slope_sum'] / meta['group_cells']
    n_groups = (
        group_df[['gene_key', 'tissue', 'cell_type']]
        .drop_duplicates()
        .groupby('gene_key', observed=True)
        .size()
        .rename('n_groups')
        .reset_index()
    )
    meta = meta.merge(n_groups, on='gene_key', how='left')
    merged = global_df.merge(meta[['gene_key', 'within_slope', 'n_groups', 'group_cells']], on='gene_key', how='inner')
    merged = merged[merged['n_groups'] >= 3].copy()
    merged = _prepare_control_frame(merged, 'slope_z', 'within_slope')
    return merged, title


def _load_residualized_celltype_control(species_label):
    _, title, species_name = _control_base(species_label)
    residualized = pd.read_csv(REPO_ROOT / 'data' / 'revision' / 'celltype_control_residualized.csv')
    merged = residualized[residualized['species'] == species_name].copy()
    merged = _prepare_control_frame(merged, 'global_slope', 'residualized_slope')
    return merged, title


def _plot_celltype_control_panel(ax, merged, title, cmap, y_label):
    values = np.abs(np.concatenate([merged['global_metric'].to_numpy(), merged['control_metric'].to_numpy()]))
    lim = np.nanquantile(values, 0.995)
    lim = max(float(lim), 0.02)
    ax.hexbin(
        merged['global_metric'],
        merged['control_metric'],
        gridsize=65,
        mincnt=1,
        bins='log',
        cmap=cmap,
        linewidths=0,
    )
    ax.axhline(0, color='0.75', linewidth=0.8)
    ax.axvline(0, color='0.75', linewidth=0.8)
    ax.plot([-lim, lim], [-lim, lim], linestyle='--', color='black', linewidth=1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Global slope (all cells)', fontsize=11)
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.set_aspect('equal', adjustable='box')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def _control_summary(species, control, merged):
    r, p = pearsonr(merged['global_metric'], merged['control_metric'])
    return {
        'species': species,
        'control': control,
        'genes': len(merged),
        'pearson_r': float(r),
        'pearson_p': float(p),
        'sign_concordance': float(merged['same_sign'].mean()),
        'median_abs_ratio': float(merged['attenuation'].median()),
    }


def _display_control_summary(stats):
    display(
        pd.DataFrame(
            [
                {
                    'species': stats['species'],
                    'control': stats['control'],
                    'genes': f"{stats['genes']:,}",
                    'pearson_r': f"{stats['pearson_r']:.3f}",
                    'pearson_p': f"{stats['pearson_p']:.2g}",
                    'sign_concordance': f"{stats['sign_concordance']:.1%}",
                    'median_abs_ratio': f"{stats['median_abs_ratio']:.3f}",
                }
            ]
        )
    )


mur_weighted, mur_title = _load_celltype_control_comparison('TabMur')
sap_weighted, sap_title = _load_celltype_control_comparison('TabSap')
mur_resid, _ = _load_residualized_celltype_control('TabMur')
sap_resid, _ = _load_residualized_celltype_control('TabSap')

plot_configs = [
    ('Mouse', 'weighted within-group', mur_weighted, f'{mur_title} | weighted within-group slopes', 'Blues', 'Weighted mean tissue x cell type slope'),
    ('Human', 'weighted within-group', sap_weighted, f'{sap_title} | weighted within-group slopes', 'Greens', 'Weighted mean tissue x cell type slope'),
    ('Mouse', 'residualized within-group', mur_resid, f'{mur_title} | residualized within-group slopes', 'Blues', 'Residualized tissue x cell type slope'),
    ('Human', 'residualized within-group', sap_resid, f'{sap_title} | residualized within-group slopes', 'Greens', 'Residualized tissue x cell type slope'),
]

summary_rows = []
for species, control, merged, title, cmap, y_label in plot_configs:
    fig, ax = plt.subplots(figsize=(6.5, 5.8), dpi=600, constrained_layout=True)
    _plot_celltype_control_panel(ax, merged, title, cmap, y_label)
    plt.show()

    stats = _control_summary(species, control, merged)
    _display_control_summary(stats)
    summary_rows.append(stats)

summary = pd.DataFrame(summary_rows)
summary['pearson_r'] = summary['pearson_r'].map(lambda x: round(float(x), 3))
summary['pearson_p'] = summary['pearson_p'].map(lambda x: f'{float(x):.2g}')
summary['sign_concordance'] = summary['sign_concordance'].map(lambda x: round(float(x), 3))
summary['median_abs_ratio'] = summary['median_abs_ratio'].map(lambda x: round(float(x), 3))
print(summary.to_string(index=False))




## Extended Data Fig. 12 - mouse GO enrichment network

This EnrichmentMap/Cytoscape artwork is not regenerated by this notebook from the reviewer package.


## Extended Data Fig. 13 - human GO enrichment network

This EnrichmentMap/Cytoscape artwork is not regenerated by this notebook from the reviewer package.


## Extended Data Fig. 14 - lineage programs across tissues


In [ ]:
tabmur_clustermap_order = _plot_clustermap(TABMUR_DIR / "expression_coupling_by_group" / "go_enrichment_by_group" / "go_combined.tsv", "A | Functional similarity across cell types (mean across tissues)")


In [ ]:
tabmur_clustermap_order_df = pd.DataFrame({
    "display_order": range(1, len(tabmur_clustermap_order) + 1),
    "cell_type": tabmur_clustermap_order,
})
print(tabmur_clustermap_order_df.to_string(index=False))


In [ ]:
tabsap_clustermap_order = _plot_clustermap(TABSAP_DIR / "expression_coupling_by_group" / "go_enrichment_by_group" / "go_combined.tsv", "A | Functional similarity across cell types (mean across tissues, TabSap)")


In [ ]:
tabsap_clustermap_order_df = pd.DataFrame({
    "display_order": range(1, len(tabsap_clustermap_order) + 1),
    "cell_type": tabsap_clustermap_order,
})
print(tabsap_clustermap_order_df.to_string(index=False))


## Extended Data Fig. 15 - cross-cell-type functional similarity


In [ ]:
_plot_chord(TABMUR_DIR / "expression_coupling_by_group" / "go_enrichment_by_group" / "go_combined.tsv", "B | Cross-tissue functional similarity among cell types (r >= 0.2)")


In [ ]:
_plot_chord(TABSAP_DIR / "expression_coupling_by_group" / "go_enrichment_by_group" / "go_combined.tsv", "", alias_map={"Muscle": "Limb_Muscle", "Bone_Marrow": "Marrow"}, with_group_labels=False)


## Additional developmental-stage support and robustness tables


In [ ]:
STAGE_WEIGHTING_LABELS = {
    "Embryoblast": "Pre-gastrulation",
    "Germ layer-specific": "Post-gastrulation",
    "Tissue-specific": "Tissue-specific",
    "Adult-specific": "Cell-type specific",
}
WEIGHTING_LABELS = {
    "Supporting reads": "Weighted",
    "Variant count": "Unweighted",
}

def _current_stage_weighting_table(species_label, agg_base, mut_csv):
    donors = _load_stage_timing_donors(agg_base)
    long_df = summarize_stage_weighting(mut_csv, donor_whitelist=donors).copy()
    summary = (
        long_df.groupby(["weighting", "Stage"], observed=True)["Percent"]
        .mean()
        .unstack("Stage")
        .reindex(columns=STAGE_ORDER)
        .rename(index=WEIGHTING_LABELS, columns=STAGE_WEIGHTING_LABELS)
        .reindex(index=["Weighted", "Unweighted"])
        .round(2)
    )
    print(f"\n{species_label}")
    print(summary.to_string())
    print(f"n_donors = {len(donors)}")
    return summary

mur_current_stage_weighting = _current_stage_weighting_table(
    "Tabula Muris",
    REPO_ROOT / "data" / "TabMur" / "aggregates",
    REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
)
sap_current_stage_weighting = _current_stage_weighting_table(
    "Tabula Sapiens",
    TABSAP_AGG,
    TABSAP_AGG / "mut_table_with_stage.csv",
)


In [ ]:
REVISION_DIR = REPO_ROOT / "data" / "revision"
SHARING_PATTERN_ORDER = [
    "Embryoblast-compatible (>1 germ layer)",
    "Germ layer-specific-compatible (1 germ layer, >1 tissue)",
    "Tissue-specific-compatible (1 germ layer, 1 tissue, >1 cell type)",
    "Adult-specific-compatible (1 germ layer, 1 tissue, 1 cell type)",
    "Ambiguous",
    "Unclassified",
]


def _classify_stage_from_breadth(n_layers, n_tissues, n_cell_types):
    if pd.isna(n_layers) or pd.isna(n_tissues) or pd.isna(n_cell_types):
        return pd.NA
    n_layers = int(n_layers)
    n_tissues = int(n_tissues)
    n_cell_types = int(n_cell_types)
    if n_layers > 1:
        return "Embryoblast"
    if n_tissues > 1:
        return "Germ layer-specific"
    if n_cell_types > 1:
        return "Tissue-specific"
    if n_layers == 1 and n_tissues == 1 and n_cell_types == 1:
        return "Adult-specific"
    return pd.NA


def _classify_sharing_pattern(stage_label, n_layers, n_tissues, n_cell_types):
    expected_stage = _classify_stage_from_breadth(n_layers, n_tissues, n_cell_types)
    if pd.isna(expected_stage):
        return "Unclassified"
    if pd.notna(stage_label) and str(stage_label) not in STAGE_ORDER:
        return "Unclassified"
    if pd.notna(stage_label) and str(stage_label) != expected_stage:
        return "Ambiguous"
    return {
        "Embryoblast": "Embryoblast-compatible (>1 germ layer)",
        "Germ layer-specific": "Germ layer-specific-compatible (1 germ layer, >1 tissue)",
        "Tissue-specific": "Tissue-specific-compatible (1 germ layer, 1 tissue, >1 cell type)",
        "Adult-specific": "Adult-specific-compatible (1 germ layer, 1 tissue, 1 cell type)",
    }[expected_stage]


def _load_within_donor_variant_breadth(mut_csv):
    cols = ["donor", "var_id", "germ_layer", "tissue", "cell_type_stage", "stage_label"]
    df = pd.read_csv(mut_csv, usecols=cols, low_memory=False)
    collapsed = (
        df.dropna(subset=["donor", "var_id", "germ_layer", "tissue", "cell_type_stage"])
        .drop_duplicates(subset=["donor", "var_id", "germ_layer", "tissue", "cell_type_stage"])
        .copy()
    )
    breadth = (
        collapsed.groupby(["donor", "var_id"], observed=True)
        .agg(
            n_layers=("germ_layer", "nunique"),
            n_tissues=("tissue", "nunique"),
            n_cell_types=("cell_type_stage", "nunique"),
        )
        .reset_index()
    )
    stage_lookup = (
        df[["donor", "var_id", "stage_label"]]
        .dropna(subset=["donor", "var_id"])
        .drop_duplicates(subset=["donor", "var_id"])
    )
    breadth = breadth.merge(stage_lookup, on=["donor", "var_id"], how="left")
    breadth["recomputed_stage"] = breadth.apply(
        lambda row: _classify_stage_from_breadth(
            row["n_layers"],
            row["n_tissues"],
            row["n_cell_types"],
        ),
        axis=1,
    )
    breadth["sharing_pattern"] = breadth.apply(
        lambda row: _classify_sharing_pattern(
            row["stage_label"],
            row["n_layers"],
            row["n_tissues"],
            row["n_cell_types"],
        ),
        axis=1,
    )
    return breadth


def _summarize_within_donor_sharing_patterns(species_label, breadth):
    summary = (
        breadth.groupby("sharing_pattern", observed=True)
        .agg(
            n_donor_variant_calls=("var_id", "size"),
            median_germ_layers=("n_layers", "median"),
            median_tissues=("n_tissues", "median"),
            median_cell_types=("n_cell_types", "median"),
        )
        .reset_index()
    )
    summary["percent_of_donor_variant_calls"] = (
        summary["n_donor_variant_calls"] / len(breadth) * 100.0
    )
    summary = pd.DataFrame({"sharing_pattern": SHARING_PATTERN_ORDER}).merge(
        summary,
        on="sharing_pattern",
        how="left",
    )
    summary.insert(0, "species", species_label)
    summary["n_donor_variant_calls"] = summary["n_donor_variant_calls"].fillna(0).astype(int)
    summary["percent_of_donor_variant_calls"] = summary["percent_of_donor_variant_calls"].fillna(0.0)
    zero_mask = summary["n_donor_variant_calls"].eq(0)
    for col in ["median_germ_layers", "median_tissues", "median_cell_types"]:
        summary.loc[zero_mask, col] = pd.NA
    return summary


def _stage_composition_by_donor_recurrence(species_label, breadth):
    per_locus = (
        breadth.groupby("var_id", observed=True)["donor"]
        .nunique()
        .rename("n_donors")
        .reset_index()
    )
    unique_calls = (
        breadth[["donor", "var_id", "stage_label"]]
        .drop_duplicates(subset=["donor", "var_id", "stage_label"])
        .merge(per_locus, on="var_id", how="left")
    )
    subset_specs = [
        ("all donor-variant calls", unique_calls),
        ("single-donor loci", unique_calls[unique_calls["n_donors"] == 1].copy()),
        ("recurrent loci (>1 donor)", unique_calls[unique_calls["n_donors"] > 1].copy()),
    ]
    rows = []
    for subset_label, sub in subset_specs:
        pct = (
            sub["stage_label"]
            .value_counts(normalize=True)
            .reindex(STAGE_ORDER, fill_value=0.0)
            .mul(100.0)
        )
        rows.append(
            {
                "species": species_label,
                "subset": subset_label,
                "n_donor_variant_calls": int(len(sub)),
                **{stage: round(float(pct[stage]), 2) for stage in STAGE_ORDER},
            }
        )
    summary = pd.DataFrame(rows)
    stage_recurrence = (
        unique_calls.assign(is_recurrent=unique_calls["n_donors"] > 1)
        .groupby("stage_label", observed=True)
        .agg(
            n_donor_variant_calls=("var_id", "size"),
            recurrent_donor_variant_calls=("is_recurrent", "sum"),
        )
        .reindex(STAGE_ORDER, fill_value=0)
        .reset_index()
    )
    stage_recurrence.insert(0, "species", species_label)
    stage_recurrence["recurrent_call_percent"] = (
        stage_recurrence["recurrent_donor_variant_calls"]
        / stage_recurrence["n_donor_variant_calls"].replace(0, np.nan)
        * 100.0
    )
    return summary, stage_recurrence


within_donor_sharing_pattern_tables = []
donor_recurrence_tables = []
stage_specific_recurrence_tables = []
for species_label, mut_csv in [
    (
        "Tabula Muris",
        REPO_ROOT / "data" / "TabMur" / "aggregates" / "mut_table_with_stage.csv",
    ),
    (
        "Tabula Sapiens",
        TABSAP_AGG / "mut_table_with_stage.csv",
    ),
]:
    breadth = _load_within_donor_variant_breadth(mut_csv)
    sharing_summary = _summarize_within_donor_sharing_patterns(species_label, breadth)
    recurrence_summary, stage_recurrence = _stage_composition_by_donor_recurrence(
        species_label,
        breadth,
    )
    within_donor_sharing_pattern_tables.append(sharing_summary)
    donor_recurrence_tables.append(recurrence_summary)
    stage_specific_recurrence_tables.append(stage_recurrence)

    print(f"\n## {species_label}: within-donor sharing-pattern audit")
    print(
        sharing_summary[
            [
                "sharing_pattern",
                "n_donor_variant_calls",
                "percent_of_donor_variant_calls",
                "median_germ_layers",
                "median_tissues",
                "median_cell_types",
            ]
        ]
        .round(
            {
                "percent_of_donor_variant_calls": 2,
                "median_germ_layers": 1,
                "median_tissues": 1,
                "median_cell_types": 1,
            }
        )
        .to_string(index=False)
    )
    print(f"\n## {species_label}: donor-recurrence sensitivity")
    print(recurrence_summary.to_string(index=False))
    print(f"\n## {species_label}: recurrent-locus fraction by stage")
    print(
        stage_recurrence.round({"recurrent_call_percent": 2}).to_string(index=False)
    )

within_donor_sharing_pattern_audit = pd.concat(
    within_donor_sharing_pattern_tables,
    ignore_index=True,
)
donor_recurrence_sensitivity = pd.concat(
    donor_recurrence_tables,
    ignore_index=True,
)
stage_specific_donor_recurrence_rates = pd.concat(
    stage_specific_recurrence_tables,
    ignore_index=True,
)
REVISION_DIR.mkdir(parents=True, exist_ok=True)
sharing_path = REVISION_DIR / "within_donor_sharing_pattern_audit.csv"
recurrence_path = REVISION_DIR / "donor_recurrence_sensitivity.csv"
stage_recurrence_path = REVISION_DIR / "stage_specific_donor_recurrence_rates.csv"
within_donor_sharing_pattern_audit.to_csv(sharing_path, index=False)
donor_recurrence_sensitivity.to_csv(recurrence_path, index=False)
stage_specific_donor_recurrence_rates.to_csv(stage_recurrence_path, index=False)
print("\nSaved revision tables:")
for path in [sharing_path, recurrence_path, stage_recurrence_path]:
    print(path.relative_to(REPO_ROOT))


In [ ]:
                                                                                                  
from scripts.revision.developmental_stage_breadth_distributions import run_stage_breadth_distribution_analysis

developmental_stage_breadth = run_stage_breadth_distribution_analysis(show_plot=True, verbose=True)
display(developmental_stage_breadth["summary_table"])


In [ ]:
support_threshold = pd.read_csv(REPO_ROOT / "data" / "revision" / "developmental_stage_support_threshold_sensitivity.csv")
callable_threshold = pd.read_csv(REPO_ROOT / "data" / "revision" / "callable_site_threshold_sensitivity.csv")
donor_details = pd.read_csv(REPO_ROOT / "data" / "revision" / "donor_filtering_details.csv")
support_threshold_long = support_threshold.melt(
    id_vars=["species", "min_supporting_cells", "n_retained_donor_variant_calls"],
    value_vars=["Embryoblast_percent", "Germ_layer_specific_percent", "Tissue_specific_percent", "Adult_specific_percent"],
    var_name="stage_label",
    value_name="variant_percent",
)
support_threshold_long["stage_label"] = support_threshold_long["stage_label"].map({
    "Embryoblast_percent": "Embryoblast",
    "Germ_layer_specific_percent": "Germ layer-specific",
    "Tissue_specific_percent": "Tissue-specific",
    "Adult_specific_percent": "Adult-specific",
})
for species in ["Tabula Muris", "Tabula Sapiens"]:
    fig, ax = plt.subplots(figsize=(8, 5), dpi=600)
    sub = support_threshold_long[support_threshold_long["species"] == species]
    for stage in STAGE_ORDER:
        stage_df = sub[sub["stage_label"] == stage]
        ax.plot(stage_df["min_supporting_cells"], stage_df["variant_percent"], marker="o", linewidth=2, color=STAGE_COLORS[stage], label=stage)
    ax.set_title(f"{species}: support-threshold sensitivity")
    ax.set_xlabel("Minimum supporting cells")
    ax.set_ylabel("Retained donor-variant composition (%)")
    ax.set_xticks(sorted(support_threshold_long["min_supporting_cells"].unique()))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    plt.show()

for species in ["Mouse", "Human"]:
    fig, ax = plt.subplots(figsize=(8, 5), dpi=600)
    sub = callable_threshold[callable_threshold["species"] == species].sort_values("min_callable_sites")
    ax.bar(sub["min_callable_sites"] / 1_000_000, sub["n_donors"], width=9.0, color="#D8D8D8", edgecolor="black", linewidth=0.4, label="Retained donors")
    ax2 = ax.twinx()
    ax2.plot(sub["min_callable_sites"] / 1_000_000, sub["spearman_rho"], marker="o", color="#123A73", linewidth=2, label="Spearman rho")
    ax.set_title(f"{species}: callable-site filtering sensitivity")
    ax.set_xlabel("Minimum callable sites (millions)")
    ax.set_ylabel("Retained donors")
    ax2.set_ylabel("Burden vs age rho")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax2.spines["top"].set_visible(False)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    plt.tight_layout()
    plt.show()
for species in ["Tabula Muris", "Tabula Sapiens"]:
    detail = support_threshold[support_threshold["species"] == species].sort_values("min_supporting_cells").copy()
    print(f"\n## {species} developmental-stage support-threshold sensitivity")
    print(
        detail[
            [
                "min_supporting_cells",
                "n_retained_donor_variant_calls",
                "Embryoblast_percent",
                "Germ_layer_specific_percent",
                "Tissue_specific_percent",
                "Adult_specific_percent",
            ]
        ]
        .rename(columns={
            "min_supporting_cells": "threshold",
            "Embryoblast_percent": "Embryoblast",
            "Germ_layer_specific_percent": "Germ layer-specific",
            "Tissue_specific_percent": "Tissue-specific",
            "Adult_specific_percent": "Adult-specific",
        })
        .to_string(index=False)
    )
    start = detail.iloc[0]
    end = detail.iloc[-1]
    shared_start = 100.0 - float(start["Adult_specific_percent"])
    shared_end = 100.0 - float(end["Adult_specific_percent"])
    print(
        f"Adult-specific rises from {start['Adult_specific_percent']:.2f}% at >= {int(start['min_supporting_cells'])} cells to {end['Adult_specific_percent']:.2f}% at >= {int(end['min_supporting_cells'])} cells; shared categories collectively fall from {shared_start:.2f}% to {shared_end:.2f}%."
    )
for species in ["Mouse", "Human"]:
    detail = donor_details[donor_details["species"] == species].copy()
    print(f"\n## {species} donor filtering")
    print(detail[["donor", "age", "in_stage_timing", "in_age_regression", "stage_timing_status", "age_regression_status"]].to_string(index=False))


In [ ]:
                                             
from scripts.revision.analyze_genome_stage_distribution import run_overlap_collision_analysis

overlap_summary, hotspot_summary, _, hover_html_path = run_overlap_collision_analysis(write_outputs=True)
top_window_summary = hotspot_summary[hotspot_summary["rank_within_stage_window_size"] == 1].copy()

display(
    overlap_summary[
        [
            "species",
            "stage_label",
            "n_unique_sites",
            "pct_unique_sites_seen_in_ge2_cells",
            "pct_unique_sites_seen_in_ge2_donors",
            "expected_shared_exact_sites_per_random_cell_pair",
            "most_recurrent_exact_site",
            "max_supporting_cells_for_exact_site",
        ]
    ]
    .rename(
        columns={
            "stage_label": "stage",
            "pct_unique_sites_seen_in_ge2_cells": "pct_sites_seen_in_ge2_cells",
            "pct_unique_sites_seen_in_ge2_donors": "pct_sites_seen_in_ge2_donors",
            "max_supporting_cells_for_exact_site": "top_site_supporting_cells",
        }
    )
    .sort_values(["species", "stage"])
    .reset_index(drop=True)
)

display(
    top_window_summary[
        [
            "species",
            "stage_label",
            "window_size_bp",
            "region",
            "n_unique_cells",
            "approx_p_two_random_cells_both_hit_window",
            "expected_shared_exact_sites_per_random_cell_pair_from_window",
            "conditional_expected_shared_exact_sites_per_cell_pair_given_both_hit_window",
            "top_exact_site_in_window",
            "top_exact_site_supporting_cells",
        ]
    ]
    .rename(
        columns={
            "stage_label": "stage",
            "approx_p_two_random_cells_both_hit_window": "p_two_random_stage_cells_both_hit_window",
            "conditional_expected_shared_exact_sites_per_cell_pair_given_both_hit_window": "expected_shared_exact_sites_per_cell_pair_given_both_hit_window",
            "top_exact_site_in_window": "top_exact_site",
            "top_exact_site_supporting_cells": "top_site_supporting_cells",
        }
    )
    .sort_values(["species", "stage", "window_size_bp"])
    .reset_index(drop=True)
)

print(f"Interactive hover browser: {hover_html_path}")
print(
    "Interpretation note: the expected shared-site rates are empirical background rates from the observed stage-specific hotspot windows; they are not posterior probabilities that any variant is an artifact."
)




In [ ]:
from pathlib import Path

import pandas as pd

stage_order = [
    "Embryoblast",
    "Germ layer-specific",
    "Tissue-specific",
    "Adult-specific",
]
rate_cols = [
    "pair_weighted_empirical_same_window_probability_10bp",
    "mean_empirical_same_window_probability_10bp",
    "median_empirical_same_window_probability_10bp",
    "pair_weighted_top_window_pair_fraction_10bp",
    "null_probability_10bp",
    "pair_weighted_empirical_same_window_probability_100bp",
    "mean_empirical_same_window_probability_100bp",
    "median_empirical_same_window_probability_100bp",
    "pair_weighted_top_window_pair_fraction_100bp",
    "null_probability_100bp",
    "pair_weighted_empirical_same_window_probability_1kb",
    "mean_empirical_same_window_probability_1kb",
    "median_empirical_same_window_probability_1kb",
    "pair_weighted_top_window_pair_fraction_1kb",
    "null_probability_1kb",
    "pair_weighted_exact_site_collision_rate",
    "mean_exact_site_collision_rate",
    "median_exact_site_collision_rate",
]

summary_df = pd.read_csv(Path("data/revision/stage_specific_same_window_clustering_summary.csv"))
summary_df["stage_label"] = pd.Categorical(
    summary_df["stage_label"],
    categories=stage_order,
    ordered=True,
)
display_df = summary_df[
    [
        "species",
        "stage_label",
        "n_donors",
        "median_n_unique_sites",
        "sum_total_site_pairs",
        "sum_same_window_pairs_10bp",
        "pair_weighted_empirical_same_window_probability_10bp",
        "mean_empirical_same_window_probability_10bp",
        "median_empirical_same_window_probability_10bp",
        "median_top_10bp_n_unique_sites",
        "sum_top_window_pairs_10bp",
        "pair_weighted_top_window_pair_fraction_10bp",
        "null_probability_10bp",
        "sum_same_window_pairs_100bp",
        "pair_weighted_empirical_same_window_probability_100bp",
        "mean_empirical_same_window_probability_100bp",
        "median_empirical_same_window_probability_100bp",
        "median_top_100bp_n_unique_sites",
        "sum_top_window_pairs_100bp",
        "pair_weighted_top_window_pair_fraction_100bp",
        "null_probability_100bp",
        "sum_same_window_pairs_1kb",
        "pair_weighted_empirical_same_window_probability_1kb",
        "mean_empirical_same_window_probability_1kb",
        "median_empirical_same_window_probability_1kb",
        "median_top_1kb_n_unique_sites",
        "sum_top_window_pairs_1kb",
        "pair_weighted_top_window_pair_fraction_1kb",
        "null_probability_1kb",
        "sum_exact_site_same_site_pairs",
        "pair_weighted_exact_site_collision_rate",
        "mean_exact_site_collision_rate",
        "median_exact_site_collision_rate",
    ]
]
display_df = display_df.sort_values(["species", "stage_label"]).reset_index(drop=True)
display_df["stage_label"] = display_df["stage_label"].astype(str)
for col in rate_cols:
    display_df[col] = display_df[col].map(lambda value: f"{value:.3e}" if pd.notna(value) else "")

print("Donor-averaged stage-specific same-window clustering probabilities (using unique donor-level mutation sites)")
display(display_df)
print(
    "Interpretation note: this table is computed from unique donor-level mutation sites, not cell-level mutation observations, so a mutation seen in multiple cells contributes once. Exact-site recurrence is therefore zero after unique-site deduplication and is shown only as a sanity check. The informative statistics are same-window clustering probabilities at 10 bp, 100 bp, and 1 kb. Top-window pair fractions should be interpreted together with the accompanying unique-site counts and total-pair denominators."
)



In [ ]:
                                                                          
from scripts.revision.developmental_stage_unified_sensitivity import run_unified_sensitivity_analysis

developmental_stage_unified_sensitivity = run_unified_sensitivity_analysis(show_plot=True, verbose=True)
display(developmental_stage_unified_sensitivity["threshold_table"])
display(developmental_stage_unified_sensitivity["retained_counts_table"])
display(developmental_stage_unified_sensitivity["substitution_table"])
display(developmental_stage_unified_sensitivity["editing_sensitivity_table"])


## Stage-label random-chance null model

This section asks whether the observed developmental-stage labels could arise by random redistribution of supporting cells. For each donor-variant, the total number of supporting cells is held fixed, then those supporting cells are reassigned across that donor's observed germ-layer/tissue/cell-type compartments with probabilities proportional to the donor's compartment cell abundance in `cb_mutation_rates__all.csv`. Stage labels are then recomputed from the randomized breadth. This is a donor-specific Monte Carlo null, not a formal SComatic per-variant p-value.


In [ ]:
NULL_STAGE_RANDOM_SPECS = [
    {
        "dataset": "TabMur",
        "species": "Mouse",
        "title": "Tabula Muris",
        "agg_base": REPO_ROOT / "data" / "TabMur" / "aggregates",
        "color": "#4C78A8",
    },
    {
        "dataset": "TabSap",
        "species": "Human",
        "title": "Tabula Sapiens",
        "agg_base": REPO_ROOT / "data" / "TabSap" / "aggregates",
        "color": "#54A24B",
    },
]
NULL_STAGE_RANDOM_N_PERMUTATIONS = 500
NULL_STAGE_RANDOM_SEED = 1729


def _null_classify_stage(n_layers, n_tissues, n_cell_types):
    if int(n_layers) > 1:
        return "Embryoblast"
    if int(n_tissues) > 1:
        return "Germ layer-specific"
    if int(n_cell_types) > 1:
        return "Tissue-specific"
    return "Adult-specific"


def _load_null_background(agg_base):
    donors = _load_stage_timing_donors(agg_base)
    bg = pd.read_csv(
        Path(agg_base) / "cb_mutation_rates__all.csv",
        usecols=["donor", "CB", "cell_type", "tissue", "germ_layer"],
        low_memory=False,
    )
    bg = bg.dropna(subset=["donor", "CB", "cell_type", "tissue", "germ_layer"]).copy()
    bg["donor"] = bg["donor"].astype(str)
    bg = bg[bg["donor"].isin(donors)].copy()
    bg = bg.rename(columns={"cell_type": "cell_type_stage"})
    bg["tissue"] = bg["tissue"].astype(str).str.strip()
    bg["germ_layer"] = bg["germ_layer"].astype(str).str.strip()
    bg["cell_type_stage"] = bg["cell_type_stage"].astype(str).str.strip()
    bg = bg.drop_duplicates(subset=["donor", "CB"])

    comp = (
        bg.groupby(["donor", "germ_layer", "tissue", "cell_type_stage"], observed=True)["CB"]
        .nunique()
        .reset_index(name="n_available_cells")
    )

    out = {}
    for donor, sub in comp.groupby("donor", observed=True):
        sub = sub.sort_values(["germ_layer", "tissue", "cell_type_stage"]).reset_index(drop=True)
        probs = sub["n_available_cells"].to_numpy(float)
        probs = probs / probs.sum()
        tissue_codes = pd.Categorical(sub["tissue"]).codes
        layer_codes = pd.Categorical(sub["germ_layer"]).codes
        comp_to_tissue = np.eye(int(tissue_codes.max()) + 1, dtype=np.int8)[tissue_codes]
        comp_to_layer = np.eye(int(layer_codes.max()) + 1, dtype=np.int8)[layer_codes]
        out[str(donor)] = {
            "probs": probs,
            "comp_to_tissue": comp_to_tissue,
            "comp_to_layer": comp_to_layer,
            "n_compartments": len(sub),
            "n_background_cells": int(sub["n_available_cells"].sum()),
        }
    return out


def _load_variant_support_for_null(agg_base):
    donors = _load_stage_timing_donors(agg_base)
    mut = pd.read_csv(
        Path(agg_base) / "mut_table_with_stage.csv",
        usecols=["donor", "CB", "var_id", "germ_layer", "tissue", "cell_type_stage", "stage_label"],
        low_memory=False,
    )
    mut = mut.dropna(subset=["donor", "CB", "var_id", "germ_layer", "tissue", "cell_type_stage"]).copy()
    mut["donor"] = mut["donor"].astype(str)
    mut = mut[mut["donor"].isin(donors)].copy()
    mut["var_id"] = mut["var_id"].astype(str)
    for col in ["germ_layer", "tissue", "cell_type_stage"]:
        mut[col] = mut[col].astype(str).str.strip()
    mut = mut.drop_duplicates(subset=["donor", "CB", "var_id"])

    presence = (
        mut.groupby(["donor", "var_id", "germ_layer", "tissue", "cell_type_stage"], observed=True)["CB"]
        .nunique()
        .reset_index(name="supporting_cells")
    )
    variant = (
        presence.groupby(["donor", "var_id"], observed=True)
        .agg(
            total_support=("supporting_cells", "sum"),
            observed_n_layers=("germ_layer", "nunique"),
            observed_n_tissues=("tissue", "nunique"),
            observed_n_cell_types=("cell_type_stage", "nunique"),
        )
        .reset_index()
    )
    variant["observed_stage"] = variant.apply(
        lambda row: _null_classify_stage(
            row["observed_n_layers"],
            row["observed_n_tissues"],
            row["observed_n_cell_types"],
        ),
        axis=1,
    )
    return variant


def _simulate_stage_null_for_species(variant_df, background_by_donor, n_permutations=500, seed=0):
    stage_to_idx = {stage: idx for idx, stage in enumerate(STAGE_ORDER)}
    idx_to_stage = {idx: stage for stage, idx in stage_to_idx.items()}

    variant_df = variant_df.copy()
    variant_df = variant_df[variant_df["donor"].astype(str).isin(background_by_donor)].reset_index(drop=True)
    obs_stage_idx = variant_df["observed_stage"].map(stage_to_idx).to_numpy(int)

    null_stage_counts = np.zeros((n_permutations, len(STAGE_ORDER)), dtype=np.int32)
    same_stage_hits = np.zeros(len(variant_df), dtype=np.int32)
    stage_hits = np.zeros((len(variant_df), len(STAGE_ORDER)), dtype=np.int16)

    rng = np.random.default_rng(seed)
    for donor, donor_positions in variant_df.groupby("donor", observed=True).indices.items():
        info = background_by_donor[str(donor)]
        donor_positions = np.asarray(donor_positions, dtype=int)
        donor_support = variant_df.iloc[donor_positions]["total_support"].to_numpy(int)
        donor_obs_stage = obs_stage_idx[donor_positions]
        comp_to_tissue = info["comp_to_tissue"]
        comp_to_layer = info["comp_to_layer"]
        probs = info["probs"]

        for support in np.sort(np.unique(donor_support)):
            if int(support) <= 0:
                continue
            mask = donor_support == int(support)
            group_positions = donor_positions[mask]
            group_obs_stage = donor_obs_stage[mask]
            n_group = len(group_positions)
            if n_group == 0:
                continue
            for perm_idx in range(n_permutations):
                counts = rng.multinomial(int(support), probs, size=n_group)
                occupied = (counts > 0).astype(np.int16)
                n_cell_types = occupied.sum(axis=1)
                n_tissues = ((occupied @ comp_to_tissue) > 0).sum(axis=1)
                n_layers = ((occupied @ comp_to_layer) > 0).sum(axis=1)

                stage_idx = np.full(n_group, stage_to_idx["Adult-specific"], dtype=np.int8)
                stage_idx[n_cell_types > 1] = stage_to_idx["Tissue-specific"]
                stage_idx[n_tissues > 1] = stage_to_idx["Germ layer-specific"]
                stage_idx[n_layers > 1] = stage_to_idx["Embryoblast"]

                null_stage_counts[perm_idx] += np.bincount(stage_idx, minlength=len(STAGE_ORDER))
                same_stage_hits[group_positions] += (stage_idx == group_obs_stage)
                np.add.at(stage_hits, (group_positions, stage_idx), 1)

    observed_counts = (
        variant_df["observed_stage"].value_counts().reindex(STAGE_ORDER, fill_value=0).astype(int)
    )
    observed_percents = observed_counts / max(len(variant_df), 1) * 100.0
    expected_counts = null_stage_counts.mean(axis=0)
    expected_percents = expected_counts / max(len(variant_df), 1) * 100.0
    expected_sd_percents = null_stage_counts.std(axis=0, ddof=1) / max(len(variant_df), 1) * 100.0

    expected_counts_safe = np.where(expected_counts <= 0, 1e-9, expected_counts)
    observed_chi2 = float(((observed_counts.to_numpy(float) - expected_counts) ** 2 / expected_counts_safe).sum())
    null_chi2 = ((null_stage_counts - expected_counts) ** 2 / expected_counts_safe).sum(axis=1)
    omnibus_p = float((1 + np.sum(null_chi2 >= observed_chi2)) / (n_permutations + 1))

    summary_rows = []
    for stage_idx, stage_label in idx_to_stage.items():
        obs_count = int(observed_counts.loc[stage_label])
        null_counts_stage = null_stage_counts[:, stage_idx]
        p_enriched = float((1 + np.sum(null_counts_stage >= obs_count)) / (n_permutations + 1))
        p_depleted = float((1 + np.sum(null_counts_stage <= obs_count)) / (n_permutations + 1))
        p_two_sided = float(min(1.0, 2.0 * min(p_enriched, p_depleted)))
        summary_rows.append(
            {
                "stage_label": stage_label,
                "observed_count": obs_count,
                "observed_percent": float(observed_percents.loc[stage_label]),
                "null_mean_count": float(expected_counts[stage_idx]),
                "null_mean_percent": float(expected_percents[stage_idx]),
                "null_sd_percent": float(expected_sd_percents[stage_idx]),
                "p_enriched_vs_random": p_enriched,
                "p_depleted_vs_random": p_depleted,
                "p_two_sided": p_two_sided,
            }
        )
    summary = pd.DataFrame(summary_rows)

    variant_probs = variant_df[["donor", "var_id", "total_support", "observed_stage"]].copy()
    for stage_idx, stage_label in idx_to_stage.items():
        slug = stage_label.lower().replace(" ", "_").replace("-", "_")
        variant_probs[f"null_prob_{slug}"] = stage_hits[:, stage_idx] / float(n_permutations)
    variant_probs["null_prob_observed_stage"] = [
        variant_probs.loc[i, f"null_prob_{stage.lower().replace(' ', '_').replace('-', '_')}"]
        for i, stage in enumerate(variant_probs["observed_stage"])
    ]

    variant_stage_summary = (
        variant_probs.groupby("observed_stage", observed=True)
        .agg(
            n_variants=("var_id", "size"),
            mean_total_support=("total_support", "mean"),
            median_total_support=("total_support", "median"),
            mean_null_prob_observed_stage=("null_prob_observed_stage", "mean"),
            median_null_prob_observed_stage=("null_prob_observed_stage", "median"),
            frac_null_prob_le_0_05=("null_prob_observed_stage", lambda x: float((x <= 0.05).mean() * 100.0)),
            frac_null_prob_le_0_01=("null_prob_observed_stage", lambda x: float((x <= 0.01).mean() * 100.0)),
        )
        .reset_index()
        .rename(columns={"observed_stage": "stage_label"})
    )

    return summary, variant_stage_summary, variant_probs, omnibus_p


random_stage_null_global_rows = []
random_stage_null_variant_stage_rows = []
random_stage_null_variant_prob_tables = []
random_stage_null_omnibus_rows = []

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=350, sharey=True)
for ax, spec_idx, spec in zip(axes, range(len(NULL_STAGE_RANDOM_SPECS)), NULL_STAGE_RANDOM_SPECS):
    background = _load_null_background(spec["agg_base"])
    variant_support = _load_variant_support_for_null(spec["agg_base"])
    summary, variant_stage_summary, variant_probs, omnibus_p = _simulate_stage_null_for_species(
        variant_support,
        background,
        n_permutations=NULL_STAGE_RANDOM_N_PERMUTATIONS,
        seed=NULL_STAGE_RANDOM_SEED + spec_idx,
    )
    summary.insert(0, "species", spec["species"])
    variant_stage_summary.insert(0, "species", spec["species"])
    variant_probs.insert(0, "species", spec["species"])

    random_stage_null_global_rows.append(summary)
    random_stage_null_variant_stage_rows.append(variant_stage_summary)
    random_stage_null_variant_prob_tables.append(variant_probs)
    random_stage_null_omnibus_rows.append(
        {
            "species": spec["species"],
            "dataset": spec["title"],
            "n_permutations": NULL_STAGE_RANDOM_N_PERMUTATIONS,
            "n_stage_timing_donors": len(background),
            "n_variants": int(len(variant_support)),
            "omnibus_stage_composition_p": omnibus_p,
        }
    )

    x = np.arange(len(STAGE_ORDER))
    width = 0.38
    ax.bar(
        x - width / 2,
        summary["observed_percent"].to_numpy(float),
        width=width,
        color=spec["color"],
        alpha=0.85,
        edgecolor="black",
        linewidth=0.5,
        label="Observed",
    )
    ax.bar(
        x + width / 2,
        summary["null_mean_percent"].to_numpy(float),
        yerr=summary["null_sd_percent"].to_numpy(float),
        width=width,
        color="#D9D9D9",
        edgecolor="black",
        linewidth=0.5,
        error_kw={"elinewidth": 1.0, "ecolor": "black", "capsize": 3.0},
        label="Randomized mean +/- SD",
    )
    ax.set_title(spec["title"])
    ax.set_ylabel("Stage fraction of donor-variant calls (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(STAGE_ORDER, rotation=30, ha="right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.3)

axes[1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle("Observed stage composition vs donor-specific random-compartment null", fontsize=16, y=1.03)
plt.tight_layout()
plt.show()

random_stage_null_global_summary = pd.concat(random_stage_null_global_rows, ignore_index=True)
random_stage_null_variant_stage_summary = pd.concat(random_stage_null_variant_stage_rows, ignore_index=True)
random_stage_null_variant_probabilities = pd.concat(random_stage_null_variant_prob_tables, ignore_index=True)
random_stage_null_omnibus = pd.DataFrame(random_stage_null_omnibus_rows)

print(
    f"Null model used {NULL_STAGE_RANDOM_N_PERMUTATIONS} permutations per species. Supporting-cell totals were held fixed per donor-variant and redistributed across donor-specific compartments weighted by background cell abundance."
)
display(random_stage_null_omnibus)
print("\nStage-level observed vs random-redistribution expectation:")
display(
    random_stage_null_global_summary.round(
        {
            "observed_percent": 2,
            "null_mean_count": 2,
            "null_mean_percent": 2,
            "null_sd_percent": 2,
            "p_enriched_vs_random": 4,
            "p_depleted_vs_random": 4,
            "p_two_sided": 4,
        }
    )
)
print("\nPer-stage probability that a donor-variant would receive its observed stage label under the random-compartment null:")
display(
    random_stage_null_variant_stage_summary.round(
        {
            "mean_total_support": 2,
            "median_total_support": 2,
            "mean_null_prob_observed_stage": 4,
            "median_null_prob_observed_stage": 4,
            "frac_null_prob_le_0_05": 2,
            "frac_null_prob_le_0_01": 2,
        }
    )
)
for species in ["Mouse", "Human"]:
    print(f"\n## {species}: lowest-probability observed stage labels under the random null")
    display(
        random_stage_null_variant_probabilities[
            random_stage_null_variant_probabilities["species"] == species
        ]
        .sort_values(["null_prob_observed_stage", "total_support", "donor", "var_id"], ascending=[True, False, True, True])
        .head(20)
        .reset_index(drop=True)
    )
